#Dependencies

In [ ]:

!pip install tensorflow
!pip install tensorboard
!pip install autokeras
!pip install keras-tuner --upgrade

In [ ]:
!pip install --quiet torch torchvision flwr numpy medmnist ray
!pip install flwr-datasets

In [ ]:
!pip install cryptography==44.0.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install cryptography==44.0.1

In [ ]:
!pip show cryptography

In [ ]:
!pip install -U "tensorflow[and-cuda]==2.19.0"

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
build_info = tf.sysconfig.get_build_info()
print("  • CUDA version:  ", build_info.get("cuda_version"))
print("  • cuDNN version: ", build_info.get("cudnn_version"))
print("GPUs visible:     ", tf.config.list_physical_devices("GPU"))


#AutoKeras Model Finder

In [ ]:
import os
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import autokeras as ak
import kerastuner
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import Callback, ModelCheckpoint

DATA_DIR  = "/content/drive/MyDrive/Colab Notebooks/tissuemnist"
MODEL_DIR = os.path.join(DATA_DIR, "models")
LOG_CSV   = os.path.join(DATA_DIR, "training_log.csv")
EPOCHS    = 20
TRIALS    = 20
os.makedirs(MODEL_DIR, exist_ok=True)

base    = Path(DATA_DIR)
x_train = np.load(base/"train_images.npy", mmap_mode="r").astype("float32")[..., None] / 255.0
y_train = np.load(base/"train_labels.npy").squeeze().astype(int)
x_val   = np.load(base/"val_images.npy",   mmap_mode="r").astype("float32")[..., None] / 255.0
y_val   = np.load(base/"val_labels.npy").squeeze().astype(int)

num_classes = 8
y_train_oh  = to_categorical(y_train, num_classes)
y_val_oh    = to_categorical(y_val,   num_classes)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

class DedupCSVLogger(Callback):
    """
    Collect rows per fit(), skip duplicate epoch callbacks, and append to CSV once.
    """
    _global_fit_counter = 0

    def __init__(self, csv_path, run_id, epochs_limit):
        super().__init__()
        self.csv_path     = csv_path
        self.run_id       = run_id
        self.epochs_limit = epochs_limit
        self.rows         = []
        self._seen        = set()
        self.fit_id       = None

    def on_train_begin(self, logs=None):
        self.fit_id = DedupCSVLogger._global_fit_counter
        DedupCSVLogger._global_fit_counter += 1
        self._seen.clear()

    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        local_epoch = epoch + 1
        key = (self.fit_id, local_epoch)
        if key in self._seen:
            return
        self._seen.add(key)

        if local_epoch > self.epochs_limit:
            return

        clean_logs = {k: float(v) for k, v in logs.items() if isinstance(v, (int, float))}
        clean_logs.update({
            "epoch":     local_epoch,
            "fit_id":    self.fit_id,
            "run_id":    self.run_id,
            "timestamp": datetime.now().isoformat()
        })
        self.rows.append(clean_logs)

    def on_train_end(self, logs=None):
        if not self.rows:
            return
        df_new = pd.DataFrame(self.rows)
        if os.path.exists(self.csv_path):
            df_old = pd.read_csv(self.csv_path)
            df_all = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_all = df_new
        df_all = df_all.drop_duplicates(subset=["run_id", "fit_id", "epoch"], keep="first")
        df_all.to_csv(self.csv_path, index=False)
        self.rows = []

ckpt_cb = ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, "epoch_{epoch:02d}.keras"),
    save_weights_only=False,
    save_freq="epoch",
)

csv_cb = DedupCSVLogger(LOG_CSV, run_id, epochs_limit=EPOCHS)


from tensorflow.keras.metrics import AUC

clf = ak.ImageClassifier(
    seed=42,
    project_name="tissuemnist_search",
    directory="/content/drive/MyDrive/Colab Notebooks/tissuemnist",
    multi_label=False,
    metrics=[
        AUC(name="auc"),
        "accuracy",
    ],
    objective=kerastuner.Objective("val_auc", "max"),
    overwrite=False,
    max_trials=TRIALS,
)


history = clf.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    callbacks=[ckpt_cb, csv_cb],
)


model = clf.export_model()
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
)

print(model.summary())
print("\nValidation eval:")
print(model.evaluate(x_val, y_val_oh, verbose=2))


#Centralized EfficientNet K-Fold

In [ ]:

import os
import sys
import json
import time
from pathlib import Path
from datetime import datetime
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import KFold
import autokeras as ak
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")


if len(tf.config.list_physical_devices('GPU')) > 0:
    gpu = tf.config.list_physical_devices('GPU')[0]
    tf.config.experimental.set_memory_growth(gpu, True)



@tf.keras.utils.register_keras_serializable(package="Custom", name="CastToFloat32")
class CastToFloat32(tf.keras.layers.Layer):
    def call(self, x):
        return tf.cast(x, tf.float32)


class Config:
    EPOCHS = 1
    N_FOLDS = 2
    BATCH_SIZE = 32
    NUM_CLASSES = 8
    LEARNING_RATE = 2e-5

    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/tissuemnist")
    MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/tissuemnist/tissuemnist_search/best_model.keras"
    RESULTS_DIR = Path("/content/drive/MyDrive/fl_image") / "single_client"
    PROGRESS_JSON_PATH = RESULTS_DIR / "training_progress.json"
    FINAL_JSON_PATH = RESULTS_DIR / "centralized_baseline_results.json"

    RESULTS_DIR.mkdir(exist_ok=True, parents=True)

print("="*60)
print("CENTRALIZED BASELINE TRAINING CONFIGURATION")
print("="*60)
print(f"Loss Function:  SparseCategoricalCrossentropy")
print(f"Optimizer:      Adam (LR={Config.LEARNING_RATE})")
print(f"Epochs:         {Config.EPOCHS} (fixed, no early stopping)")
print(f"Batch Size:     {Config.BATCH_SIZE}")
print(f"Folds:          {Config.N_FOLDS}")
print(f"Classes:        {Config.NUM_CLASSES}")
print(f"Persistence:    JSON-only (epoch-level saving)")
print("="*60)


def numpy_to_list(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer, np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, dict):
        return {key: numpy_to_list(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [numpy_to_list(item) for item in obj]
    else:
        return obj

def save_json_progress(master_results, fold_results=None, message=""):

    master_results["last_update"] = datetime.now().isoformat()

    if fold_results and any(fold_results.values()):
            current_stats = {}
            for metric_name, values in fold_results.items():
                if values:
                    current_stats[metric_name] = {
                        "mean": float(np.mean(values)),
                        "std": float(np.std(values)) if len(values) > 1 else 0.0,
                        "min": float(np.min(values)),
                        "max": float(np.max(values)),
                        "values": numpy_to_list(values),
                        "n_folds_completed": len(values)
                    }

            master_results["summary_statistics"] = current_stats

            if fold_results.get('val_acc') and fold_results['val_acc']:
                best_fold_idx = np.argmax(fold_results['val_acc'])
                best_fold_num = best_fold_idx + 1

                master_results["best_fold_info"] = {
                    "fold_number": int(best_fold_num),
                    "val_accuracy": float(fold_results['val_acc'][best_fold_idx]),
                    "val_auc": float(fold_results['val_auc'][best_fold_idx]) if 'val_auc' in fold_results and fold_results['val_auc'] else None,
                    "test_accuracy": float(fold_results['test_acc'][best_fold_idx]) if 'test_acc' in fold_results and fold_results['test_acc'] else None,
                    "test_auc": float(fold_results['test_auc'][best_fold_idx]) if 'test_auc' in fold_results and fold_results['test_auc'] else None
                }

    with open(Config.PROGRESS_JSON_PATH, 'w') as f:
            json.dump(master_results, f, indent=2)

    with open(Config.FINAL_JSON_PATH, 'w') as f:
            json.dump(master_results, f, indent=2)

    if message:
            print(f" {message}")

    return True



def load_progress():
    if Config.PROGRESS_JSON_PATH.exists():
        try:
            with open(Config.PROGRESS_JSON_PATH, 'r') as f:
                return json.load(f)
        except Exception as e:
            print(f"Error loading progress: {e}")
            return None
    return None


class EpochLevelJSONCallback(tf.keras.callbacks.Callback):
    def __init__(self, master_results, fold_results, fold_num):
        super().__init__()
        self.master_results = master_results
        self.fold_results = fold_results
        self.fold_num = fold_num
        self.fold_key = f"fold_{fold_num}"
        self.epoch_start_time = None
        self.training_start_time = time.time()

        if self.fold_key not in self.master_results["fold_histories"]:
            self.master_results["fold_histories"][self.fold_key] = {
                "epochs": [],
                "train_loss": [],
                "train_accuracy": [],
                "val_loss": [],
                "val_accuracy": [],
                "current_epoch": 0,
                "total_epochs": Config.EPOCHS,
                "status": "training"
            }

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start_time = time.time()
        print(f"\n  Epoch {epoch + 1}/{Config.EPOCHS} starting...")

    def on_epoch_end(self, epoch, logs=None):
        epoch_time = time.time() - self.epoch_start_time
        total_time = time.time() - self.training_start_time

        fold_history = self.master_results["fold_histories"][self.fold_key]

        fold_history["epochs"].append(epoch + 1)
        fold_history["train_loss"].append(float(logs.get('loss', 0)))
        fold_history["train_accuracy"].append(float(logs.get('accuracy', 0)))
        fold_history["val_loss"].append(float(logs.get('val_loss', 0)))
        fold_history["val_accuracy"].append(float(logs.get('val_accuracy', 0)))
        fold_history["current_epoch"] = epoch + 1
        fold_history["last_epoch_time_seconds"] = epoch_time
        fold_history["total_time_minutes"] = total_time / 60

        self.master_results["current_fold"] = self.fold_num
        self.master_results["current_epoch"] = epoch + 1
        self.master_results["training_status"] = f"Fold {self.fold_num}/{Config.N_FOLDS}, Epoch {epoch + 1}/{Config.EPOCHS}"

        print(f"  Epoch {epoch + 1}/{Config.EPOCHS} completed in {epoch_time:.1f}s")
        print(f"    Loss: {logs.get('loss', 0):.4f} | Acc: {logs.get('accuracy', 0):.4f}")
        print(f"    Val Loss: {logs.get('val_loss', 0):.4f} | Val Acc: {logs.get('val_accuracy', 0):.4f}")
        print(f"    Total elapsed: {total_time/60:.1f} minutes")

        save_json_progress(
            self.master_results,
            self.fold_results,
            f"Saved after Fold {self.fold_num}, Epoch {epoch + 1}/{Config.EPOCHS}"
        )

    def on_train_batch_end(self, batch, logs=None):
        if batch % 100 == 0 and batch > 0:
            print(f"    Batch {batch}: loss={logs.get('loss', 0):.4f}, acc={logs.get('accuracy', 0):.4f}")


def load_and_prepare_data():


    x_train = np.load(Config.DATA_DIR/"train_images.npy").astype("float32")
    x_val = np.load(Config.DATA_DIR/"val_images.npy").astype("float32")
    x_test = np.load(Config.DATA_DIR/"test_images.npy").astype("float32")

    if len(x_train.shape) == 3:
        x_train = np.expand_dims(x_train, -1)
    if len(x_val.shape) == 3:
        x_val = np.expand_dims(x_val, -1)
    if len(x_test.shape) == 3:
        x_test = np.expand_dims(x_test, -1)

    x_train = x_train / 255.0
    x_val = x_val / 255.0
    x_test = x_test / 255.0

    y_train_raw = np.load(Config.DATA_DIR/"train_labels.npy")
    y_val_raw = np.load(Config.DATA_DIR/"val_labels.npy")
    y_test_raw = np.load(Config.DATA_DIR/"test_labels.npy")

    print(f"Raw label shape: {y_train_raw.shape}")
    print(f"First label sample: {y_train_raw[0]}")

    if len(y_train_raw.shape) == 2 and y_train_raw.shape[1] == Config.NUM_CLASSES:
        print(" Detected ONE-HOT labels, converting to sparse integers...")

        y_train = np.argmax(y_train_raw, axis=1).astype(np.int32)
        y_val = np.argmax(y_val_raw, axis=1).astype(np.int32)
        y_test = np.argmax(y_test_raw, axis=1).astype(np.int32)

        y_train_oh = y_train_raw.astype("float32")
        y_val_oh = y_val_raw.astype("float32")
        y_test_oh = y_test_raw.astype("float32")

    else:

        y_train = y_train_raw.squeeze().astype(np.int32)
        y_val = y_val_raw.squeeze().astype(np.int32)
        y_test = y_test_raw.squeeze().astype(np.int32)

        y_train_oh = to_categorical(y_train, Config.NUM_CLASSES)
        y_val_oh = to_categorical(y_val, Config.NUM_CLASSES)
        y_test_oh = to_categorical(y_test, Config.NUM_CLASSES)

    print(f" Sparse label shape: {y_train.shape}")
    print(f" Sample sparse labels: {y_train[:5]}")
    print(f" Label range: [{y_train.min()}, {y_train.max()}]")
    print(f" Training samples: {len(x_train):,}")
    print(f" Validation samples: {len(x_val):,}")
    print(f" Test samples: {len(x_test):,}")

    total_size_gb = (x_train.nbytes + x_val.nbytes + x_test.nbytes) / (1024**3)
    print(f" Total data size: {total_size_gb:.2f} GB")

    return {
        'x_train': x_train, 'y_train': y_train, 'y_train_oh': y_train_oh,
        'x_val': x_val, 'y_val': y_val, 'y_val_oh': y_val_oh,
        'x_test': x_test, 'y_test': y_test, 'y_test_oh': y_test_oh
    }

data = load_and_prepare_data()
x_train, y_train, y_train_oh = data['x_train'], data['y_train'], data['y_train_oh']
x_val, y_val, y_val_oh = data['x_val'], data['y_val'], data['y_val_oh']
x_test, y_test, y_test_oh = data['x_test'], data['y_test'], data['y_test_oh']

previous_progress = load_progress()
completed_folds = []

if previous_progress and previous_progress.get('completed_folds'):


    resume = input("\nResume from previous progress? (y/n): ").strip().lower() == 'y'
    if resume:

        master_results = previous_progress
        completed_folds = previous_progress.get('completed_folds', [])

        fold_results = {
            'train_acc': [], 'train_auc': [],
            'val_acc': [], 'val_auc': [],
            'test_acc': [], 'test_auc': []
        }

        for fold_num in completed_folds:
            fold_key = f"fold_{fold_num}"
            if fold_key in master_results.get("fold_metrics", {}):
                metrics = master_results["fold_metrics"][fold_key].get("final_metrics", {})
                if metrics:
                    fold_results['train_acc'].append(metrics.get('train_accuracy', 0))
                    fold_results['train_auc'].append(metrics.get('train_auc', 0))
                    fold_results['val_acc'].append(metrics.get('val_accuracy', 0))
                    fold_results['val_auc'].append(metrics.get('val_auc', 0))
                    fold_results['test_acc'].append(metrics.get('test_accuracy', 0))
                    fold_results['test_auc'].append(metrics.get('test_auc', 0))
    else:
        print("Starting fresh training...")
        previous_progress = None
        completed_folds = []
else:
    print("\n Starting new training session")
    completed_folds = []


if previous_progress is None:
    master_results = {
        "experiment_type": "centralized_baseline",
        "fl_comparison": {
            "centralized": "1 client × 20 epochs × 100% data",
            "federated": "10 clients × 2 epochs/round × 10 rounds"
        },
        "training_config": {
            "epochs": Config.EPOCHS,
            "epochs_fixed": True,
            "early_stopping": False,
            "lr_scheduling": False,
            "n_folds": Config.N_FOLDS,
            "batch_size": Config.BATCH_SIZE,
            "num_classes": Config.NUM_CLASSES,
            "learning_rate": Config.LEARNING_RATE,
            "optimizer": "Adam",
            "loss_function": "SparseCategoricalCrossentropy",
            "model_path": str(Config.MODEL_PATH),
            "timestamp": datetime.now().isoformat(),
            "persistence": "JSON-only with epoch-level saving"
        },
        "data_info": {
            "train_shape": list(x_train.shape),
            "val_shape": list(x_val.shape),
            "test_shape": list(x_test.shape),
            "train_samples": len(x_train),
            "val_samples": len(x_val),
            "test_samples": len(x_test)
        },
        "fold_histories": {},
        "fold_metrics": {},
        "fold_confusion_matrices": {},
        "summary_statistics": {},
        "final_evaluation": {},
        "best_fold_info": {},
        "completed_folds": [],
        "current_fold": 0,
        "current_epoch": 0,
        "training_status": "initialized"
    }

    fold_results = {
        'train_acc': [], 'train_auc': [],
        'val_acc': [], 'val_auc': [],
        'test_acc': [], 'test_auc': []
    }


kfold = KFold(n_splits=Config.N_FOLDS, shuffle=True, random_state=42)

print(f"\n{'='*60}")
print(f"STARTING {Config.N_FOLDS}-FOLD CROSS-VALIDATION")
print(f"{'='*60}")
print(f"Protocol:")
print(f"  • {Config.N_FOLDS} folds")
print(f"  • {Config.EPOCHS} epochs per fold (fixed)")
print(f"  • No early stopping")
print(f"  • JSON saved after EVERY epoch")
print(f"  • Progress file: {Config.PROGRESS_JSON_PATH.name}")
print(f"{'='*60}\n")


for fold_num, (train_idx, val_idx) in enumerate(kfold.split(x_train), 1):

    if fold_num in completed_folds:
        print(f" Fold {fold_num} already completed, skipping...")
        continue

    print(f"\n{'='*50}")
    print(f"FOLD {fold_num}/{Config.N_FOLDS}")
    print(f"{'='*50}")

    fold_key = f"fold_{fold_num}"
    if fold_key not in master_results["fold_metrics"]:
        master_results["fold_metrics"][fold_key] = {}
    if fold_key not in master_results["fold_confusion_matrices"]:
        master_results["fold_confusion_matrices"][fold_key] = {}

    x_fold_train = x_train[train_idx]
    y_fold_train = y_train[train_idx]
    x_fold_val = x_train[val_idx]
    y_fold_val = y_train[val_idx]


    y_fold_train_oh = y_train_oh[train_idx]
    y_fold_val_oh = y_train_oh[val_idx]

    print(f"Data: {len(x_fold_train):,} train, {len(x_fold_val):,} validation")


    steps_per_epoch = len(x_fold_train) // Config.BATCH_SIZE
    validation_steps = len(x_fold_val) // Config.BATCH_SIZE
    print(f"Steps: {steps_per_epoch} per epoch, {validation_steps} validation")


    master_results["fold_metrics"][fold_key]["data_split"] = {
        "train_indices_count": len(train_idx),
        "val_indices_count": len(val_idx),
        "steps_per_epoch": steps_per_epoch,
        "validation_steps": validation_steps
    }

    print(f"\nLoading model: {Path(Config.MODEL_PATH).name}")

    custom_objects = {
        'CastToFloat32': CastToFloat32,
        'Custom>CastToFloat32': CastToFloat32,
        'autokeras.keras_layers.CastToFloat32': CastToFloat32
    }


    import autokeras as ak
    if hasattr(ak, 'CUSTOM_OBJECTS'):
            custom_objects.update(ak.CUSTOM_OBJECTS)



    model = tf.keras.models.load_model(Config.MODEL_PATH, custom_objects=custom_objects, compile=False)
    print(f" Model loaded successfully")

    print(f"Model parameters: {model.count_params():,}")


    model.compile(
        optimizer=tf.keras.optimizers.Adam(Config.LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=["accuracy"],
        jit_compile=False
    )

    print(f" Model compiled (SparseCategoricalCrossentropy, LR={Config.LEARNING_RATE})")


    master_results["current_fold"] = fold_num
    master_results["training_status"] = f"Starting Fold {fold_num}/{Config.N_FOLDS}"


    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=Config.RESULTS_DIR / f"fold_{fold_num}_best_model.keras",
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1
        ),
        EpochLevelJSONCallback(master_results, fold_results, fold_num),
        tf.keras.callbacks.TensorBoard(
            log_dir=Config.RESULTS_DIR / f"logs/fold_{fold_num}",
            update_freq='epoch'
        )
    ]

    print(f"\n Training fold {fold_num} for {Config.EPOCHS} epochs...")
    print(f"   Starting at {datetime.now().strftime('%H:%M:%S')}")
    print(f"   JSON will be saved after each epoch")


    training_start = time.time()

    history = model.fit(
            x_fold_train, y_fold_train,
            batch_size=Config.BATCH_SIZE,
            epochs=Config.EPOCHS,
            validation_data=(x_fold_val, y_fold_val),
            callbacks=callbacks,
            verbose=1,
        )

    training_time = time.time() - training_start

    master_results["fold_histories"][fold_key]["status"] = "completed"
    master_results["fold_histories"][fold_key]["training_time_minutes"] = training_time/60



    print(f"\nEvaluating fold {fold_num}...")

    eval_start = time.time()

    pred_batch_size = min(Config.BATCH_SIZE, 16)

    y_fold_train_pred = model.predict(x_fold_train, verbose=1, batch_size=pred_batch_size)
    y_fold_val_pred = model.predict(x_fold_val, verbose=1, batch_size=pred_batch_size)
    y_test_pred = model.predict(x_test, verbose=1, batch_size=pred_batch_size)

    eval_time = time.time() - eval_start
    print(f"Evaluation completed in {eval_time/60:.1f} minutes")

    y_fold_train_hat = np.argmax(y_fold_train_pred, axis=1)
    y_fold_val_hat = np.argmax(y_fold_val_pred, axis=1)
    y_test_hat = np.argmax(y_test_pred, axis=1)

    train_acc = accuracy_score(y_fold_train, y_fold_train_hat)
    val_acc = accuracy_score(y_fold_val, y_fold_val_hat)
    test_acc = accuracy_score(y_test, y_test_hat)

    train_auc = roc_auc_score(y_fold_train_oh, y_fold_train_pred, multi_class="ovr", average="macro")
    val_auc = roc_auc_score(y_fold_val_oh, y_fold_val_pred, multi_class="ovr", average="macro")
    test_auc = roc_auc_score(y_test_oh, y_test_pred, multi_class="ovr", average="macro")

    fold_results['train_acc'].append(train_acc)
    fold_results['train_auc'].append(train_auc)
    fold_results['val_acc'].append(val_acc)
    fold_results['val_auc'].append(val_auc)
    fold_results['test_acc'].append(test_acc)
    fold_results['test_auc'].append(test_auc)

    print(f"\nFold {fold_num} Results:")
    print(f"  Train: Acc={train_acc:.4f}, AUC={train_auc:.4f}")
    print(f"  Val:   Acc={val_acc:.4f}, AUC={val_auc:.4f}")
    print(f"  Test:  Acc={test_acc:.4f}, AUC={test_auc:.4f}")

    master_results["fold_metrics"][fold_key]["final_metrics"] = {
        "train_accuracy": float(train_acc),
        "train_auc": float(train_auc),
        "val_accuracy": float(val_acc),
        "val_auc": float(val_auc),
        "test_accuracy": float(test_acc),
        "test_auc": float(test_auc),
        "evaluation_time_minutes": eval_time/60
    }

    completed_folds.append(fold_num)
    master_results["completed_folds"] = completed_folds
    master_results["training_status"] = f"Completed Fold {fold_num}/{Config.N_FOLDS}"

    save_json_progress(master_results, fold_results, f"Completed fold {fold_num}/{Config.N_FOLDS}")

    backup_path = Config.RESULTS_DIR / f"fold_{fold_num}_complete_backup.json"
    with open(backup_path, 'w') as f:
        json.dump(master_results, f, indent=2)


    del model
    tf.keras.backend.clear_session()

    import gc
    gc.collect()



print(f"\n{'='*60}")
print("CENTRALIZED TRAINING COMPLETE")
print(f"{'='*60}")

summary_stats = {}
for metric_name, values in fold_results.items():
    if values:
        mean_val = np.mean(values)
        std_val = np.std(values)
        min_val = np.min(values)
        max_val = np.max(values)

        summary_stats[metric_name] = {
            "mean": float(mean_val),
            "std": float(std_val),
            "min": float(min_val),
            "max": float(max_val),
            "values": numpy_to_list(values),
            "n_folds": len(values)
        }

        print(f"{metric_name.upper():12s}: {mean_val:.4f} ± {std_val:.4f}")

master_results["summary_statistics"] = summary_stats
master_results["training_status"] = "completed"

if fold_results['val_acc']:
    best_fold_idx = np.argmax(fold_results['val_acc'])
    best_fold_num = best_fold_idx + 1

    master_results["best_fold_info"] = {
        "fold_number": int(best_fold_num),
        "val_accuracy": float(fold_results['val_acc'][best_fold_idx]),
        "val_auc": float(fold_results['val_auc'][best_fold_idx]),
        "test_accuracy": float(fold_results['test_acc'][best_fold_idx]),
        "test_auc": float(fold_results['test_auc'][best_fold_idx])
    }


    print(f"   Val Acc: {fold_results['val_acc'][best_fold_idx]:.4f}")
    print(f"   Val AUC: {fold_results['val_auc'][best_fold_idx]:.4f}")

save_json_progress(master_results, fold_results, "Training complete - final save")

comparison_data = {
    "experiment": "Centralized vs Federated Learning",
    "timestamp": datetime.now().isoformat(),

    "centralized_baseline": {
        "setup": {
            "clients": 1,
            "data": "100% (full dataset)",
            "epochs": Config.EPOCHS,
            "batch_size": Config.BATCH_SIZE,
            "learning_rate": Config.LEARNING_RATE,
            "loss": "SparseCategoricalCrossentropy",
            "optimizer": "Adam"
        },
        "results": summary_stats if summary_stats else {},
        "best_fold": master_results.get("best_fold_info", {})
    },

    "federated_learning": {
        "setup": {
            "clients": 10,
            "data_per_client": "~10%",
            "epochs_per_round": 2,
            "rounds": 10,
            "total_epochs": 20,
            "batch_size": 32,
            "learning_rate": "2e-5",
            "loss": "SparseCategoricalCrossentropy",
            "aggregation": "FedAvg"
        },
        "results": {
            "note": "Add federated results here"
        }
    }
}

comparison_path = Config.RESULTS_DIR / "centralized_vs_federated_comparison.json"
with open(comparison_path, 'w') as f:
    json.dump(comparison_data, f, indent=2)
print(f" Comparison template saved to: {comparison_path}")

print(f"\n{'='*60}")
print(" TRAINING COMPLETE!")
print(f"{'='*60}")

if summary_stats:
    print(f"\n Centralized Results (Mean ± Std across {Config.N_FOLDS} folds):")
    print(f"  Val Accuracy:  {summary_stats['val_acc']['mean']:.2%} ± {summary_stats['val_acc']['std']:.2%}")
    print(f"  Val AUC:       {summary_stats['val_auc']['mean']:.4f} ± {summary_stats['val_auc']['std']:.4f}")
    print(f"  Test Accuracy: {summary_stats['test_acc']['mean']:.2%} ± {summary_stats['test_acc']['std']:.2%}")
    print(f"  Test AUC:      {summary_stats['test_auc']['mean']:.4f} ± {summary_stats['test_auc']['std']:.4f}")


#FL EfficientNet B7 IID

In [ ]:
import os
import glob
import json
import site
import gc
import sys
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional, Tuple, List, Dict, Any
import numpy as np


def is_oom_error(error_msg: str) -> bool:
    oom_indicators = [
        "OutOfMemoryError",
        "killed due to the node running low on memory",
        "memory pressure (OOM)",
        "ray.exceptions.OutOfMemoryError",
        "Task was killed due to memory",
        "running low on memory"
    ]

    error_str = str(error_msg).lower()
    return any(indicator.lower() in error_str for indicator in oom_indicators)


class FLConfig:

    NUM_CLIENTS = 2
    TOTAL_ROUNDS = 3
    LOCAL_EPOCHS = 2
    BATCH_SIZE = 32

    DEBUG_LIMIT_TR = 30
    DEBUG_LIMIT_VA = 20

    SPLIT_SEED = 42

    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/tissuemnist")
    MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/tissuemnist/tissuemnist_search/best_model.keras"
    DRIVE_DIR = Path("/content/drive/MyDrive/fl_image/fl_eff_iid")
    REPORTS_DIR = DRIVE_DIR / "reports"

    SERVER_STATE_PATH = DRIVE_DIR / "server_state.json"
    GLOBAL_LATEST_NPZ = DRIVE_DIR / "global_latest.npz"
    GLOBAL_LATEST_KERAS = DRIVE_DIR / "global_latest.keras"
    GLOBAL_BEST_NPZ = DRIVE_DIR / "global_best.npz"
    GLOBAL_BEST_KERAS = DRIVE_DIR / "global_best.keras"
    GLOBAL_BEST_META = DRIVE_DIR / "global_best_meta.json"

    CLIENT_LAST_DIR = DRIVE_DIR / "client_last_models"

    CKPT_FILE = DRIVE_DIR / "server_ckpt.npz"
    META_FILE = DRIVE_DIR / "server_ckpt_meta.json"


def setup_environment():
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "0")
    os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
    os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
    os.environ.setdefault("TF_ENABLE_LAYOUT_OPTIMIZER", "0")
    os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

    libdirs = []
    for d in site.getsitepackages():
        libdirs += glob.glob(os.path.join(d, "nvidia", "**", "lib"), recursive=True)

    seen, out = set(), []
    for p in libdirs:
        if os.path.isdir(p) and p not in seen:
            seen.add(p)
            out.append(p)

    if out:
        cuda_path = ":".join(out)
        os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

setup_environment()


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable
import flwr as fl
from flwr.client import NumPyClient
from flwr.server import ServerConfig
from flwr.common import Context
from flwr.server.strategy import FedAvg
from flwr.common import FitIns, EvaluateIns, ndarrays_to_parameters, parameters_to_ndarrays


def configure_tensorflow():
    for gpu in tf.config.list_physical_devices("GPU"):

        tf.config.experimental.set_memory_growth(gpu, True)



        tf.config.optimizer.set_jit(False)
        tf.config.optimizer.set_experimental_options({
            "layout_optimizer": False,
            "remapping": False,
            "arithmetic_optimization": False,
            "constant_folding": False,
            "disable_meta_optimizer": True,
        })


configure_tensorflow()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def save_ndarrays_npz(path: Path, arrays: List[np.ndarray]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{f"arr_{i}": a for i, a in enumerate(arrays)})

def load_ndarrays_npz(path: Path) -> List[np.ndarray]:
    with np.load(path, allow_pickle=False) as d:
        keys = sorted([k for k in d.files if k.startswith("arr_")],
                     key=lambda k: int(k.split("_")[1]))
        return [d[k] for k in keys]

def write_server_state(last_round: int, best_accuracy: float = None, best_round: int = None) -> None:
    payload = {"last_round": int(last_round), "updated_at": now_iso()}

    if best_accuracy is not None:
        payload["best_accuracy"] = float(best_accuracy)
    if best_round is not None:
        payload["best_round"] = int(best_round)


        if FLConfig.SERVER_STATE_PATH.exists():
            with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                old = json.load(f)
                if isinstance(old, dict):
                    old.update(payload)
                    payload = old


    with open(FLConfig.SERVER_STATE_PATH, "w") as f:
        json.dump(payload, f, indent=2)

def read_last_round() -> int:
    if FLConfig.SERVER_STATE_PATH.exists():

            with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                st = json.load(f)
                return int(st.get("last_round", 0))

    return 0

def read_best_model_info() -> Tuple[float, int]:
    if FLConfig.SERVER_STATE_PATH.exists():

            with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                st = json.load(f)
                return float(st.get("best_accuracy", 0.0)), int(st.get("best_round", 0))

    return 0.0, 0


@register_keras_serializable(package="Custom", name="CastToFloat32")
class CastToFloat32(keras.layers.Layer):
    def call(self, x):
        return tf.cast(x, tf.float32)

def get_autokeras_custom_objects() -> Dict:
    objs = {
        "Custom>CastToFloat32": CastToFloat32,
        "CastToFloat32": CastToFloat32,
        "autokeras.keras_layers.CastToFloat32": CastToFloat32,
    }


    import autokeras as ak
    if hasattr(ak, "CUSTOM_OBJECTS") and isinstance(ak.CUSTOM_OBJECTS, dict):
            objs.update(ak.CUSTOM_OBJECTS)


    return objs

def load_autokeras_model_safely(model_path: str) -> keras.Model:
    model = keras.models.load_model(
        model_path,
        custom_objects=get_autokeras_custom_objects(),
        compile=False
    )

    model.compile(
        optimizer=keras.optimizers.Adam(2e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=["accuracy"],
        jit_compile=False,
    )

    return model


def maybe_expand_channels(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 3:
        arr = arr[..., None]
    return arr

def load_numpy_tissuemnist(data_dir: Path) -> Tuple:
    Xtr = np.load(data_dir / "train_images.npy", mmap_mode="r")
    ytr = np.load(data_dir / "train_labels.npy").squeeze().astype(np.int32)
    Xva = np.load(data_dir / "val_images.npy", mmap_mode="r")
    yva = np.load(data_dir / "val_labels.npy").squeeze().astype(np.int32)

    Xtr = maybe_expand_channels(Xtr)
    Xva = maybe_expand_channels(Xva)


    Xte_path = data_dir / "test_images.npy"
    yte_path = data_dir / "test_labels.npy"
    Xte = yte = None

    if Xte_path.exists() and yte_path.exists():
        Xte = maybe_expand_channels(np.load(Xte_path, mmap_mode="r"))
        yte = np.load(yte_path).squeeze().astype(np.int32)

    return (Xtr, ytr), (Xva, yva), (Xte, yte)

def make_equal_class_splits(y: np.ndarray, num_clients: int, seed: int = 42) -> List[np.ndarray]:

    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    classes = np.unique(y)

    client_indices = [[] for _ in range(num_clients)]



    for cls in classes:
        cls_indices = np.where(y == cls)[0]

        rng.shuffle(cls_indices)

        n_samples = len(cls_indices)
        samples_per_client = n_samples // num_clients
        remainder = n_samples % num_clients
        start_idx = 0
        for client_id in range(num_clients):

            end_idx = start_idx + samples_per_client


            if client_id < remainder:
                end_idx += 1

            if start_idx < len(cls_indices):
                client_indices[client_id].extend(cls_indices[start_idx:end_idx])

            start_idx = end_idx


    result = []
    for client_id in range(num_clients):
        indices = np.array(client_indices[client_id], dtype=int)
        rng.shuffle(indices)
        result.append(indices)
        client_classes, client_counts = np.unique(y[indices], return_counts=True)
        class_dist = dict(zip(client_classes, client_counts))
    return result


def ensure_equal_class_files(y_train: np.ndarray, num_clients: int, seed: int, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    need = any(not (outdir / f"train_idx_client{cid}.npy").exists()
              for cid in range(num_clients))

    if need:
        parts = make_equal_class_splits(y_train, num_clients, seed)
        for cid, idx in enumerate(parts):
            np.save(outdir / f"train_idx_client{cid}.npy", idx)

def make_tf_datasets(Xtr: np.ndarray, ytr: np.ndarray,
                    Xva: np.ndarray, yva: np.ndarray,
                    H: int, W: int, Cwant: int, batch_size: int,  seed: int = 42) -> Tuple:
    AUTOTUNE = tf.data.AUTOTUNE

    @tf.autograph.experimental.do_not_convert
    def preprocess(img, label):
        img = tf.image.resize(img, (H, W))
        img = tf.cast(img, tf.float32) / 255.0
        c_in = tf.shape(img)[-1]
        img = tf.cond(
            tf.logical_and(tf.equal(Cwant, 3), tf.equal(c_in, 1)),
            lambda: tf.tile(img, [1, 1, 3]),
            lambda: img,
        )
        return img, tf.cast(label, tf.int32)

    def build(x, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        if shuffle:
            ds = ds.shuffle(min(8192, y.shape[0]), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(batch_size)
        ds = ds.cache().prefetch(AUTOTUNE)
        return ds

    return build(Xtr, ytr, True), build(Xva, yva, False)



class PrintMetrics(keras.callbacks.Callback):

    def __init__(self, cid: str, server_round: int):
        super().__init__()
        self.cid = str(cid)
        self.server_round = int(server_round)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        a = float(logs.get("accuracy", 0.0))
        l = float(logs.get("loss", 0.0))
        va = float(logs.get("val_accuracy", 0.0))
        vl = float(logs.get("val_loss", 0.0))

        print(
            f"[CID {self.cid}] r{self.server_round} epoch {epoch+1} "
            f"acc={a:.4f} loss={l:.4f} val_acc={va:.4f} val_loss={vl:.4f}",
            flush=True
        )

class SaveHistory(keras.callbacks.Callback):

    def __init__(self, json_path: Path, cid: str, server_round: int):
        super().__init__()
        self.json_path = Path(json_path)
        self.cid = str(cid)
        self.server_round = int(server_round)


        self.payload = {"cid": self.cid, "history": []}
        if self.json_path.exists():

                with open(self.json_path, "r") as f:
                    data = json.load(f)
                    if isinstance(data, dict):
                        self.payload.update(data)


        self.payload.setdefault("created_at", now_iso())
        self.payload["updated_at"] = now_iso()

    def on_train_begin(self, logs=None):
        self.payload["run_started_at"] = now_iso()
        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        row = {
            "timestamp": now_iso(),
            "server_round": self.server_round,
            "epoch": int(epoch + 1),
            "scope": "local",
        }

        for k, v in logs.items():
            if isinstance(v, (int, float, np.floating)):
                row[k] = float(v)

        self.payload["history"].append(row)
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_train_end(self, logs=None):
        self.payload["run_finished_at"] = now_iso()
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

class LastModelSaver(keras.callbacks.Callback):
    def __init__(self, cid: int):
        super().__init__()
        self.cid = int(cid)


        FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)


        self.last_path = FLConfig.CLIENT_LAST_DIR / f"client_{self.cid}_last.keras"

    def on_epoch_end(self, epoch, logs=None):
        self.model.save(self.last_path, overwrite=True)


class TFClient(NumPyClient):

    def __init__(self):

        configure_tensorflow()

        (self.Xtr_all, self.ytr_all), (self.Xva_all, self.yva_all), _ = \
            load_numpy_tissuemnist(FLConfig.DATA_DIR)

        self.model = load_autokeras_model_safely(FLConfig.MODEL_PATH)

        try:
            _, H, W, C = self.model.input_shape
        except Exception:
            H = W = 28
            C = 1

        self.input_H = int(H)
        self.input_W = int(W)
        self.input_C = int(C)


        self.cid = None
        self.train_ds = None
        self.val_ds = None
        self.n_train = 0
        self.n_val = 0
        self.hist_path = None
        self.eval_path = None
        self.last_local_weights = None

    def _prepare_for_cid(self, cid: int):
        if self.cid == cid and self.train_ds is not None:
            return
        split_dir = FLConfig.DATA_DIR / f"splits_equal_class_seed{FLConfig.SPLIT_SEED}"
        idx_path = split_dir / f"train_idx_client{cid}.npy"

        if idx_path.exists():
            idx = np.load(idx_path)
        else:
            idx = np.arange(len(self.ytr_all))

        rng = np.random.default_rng(FLConfig.SPLIT_SEED + int(cid))

        if FLConfig.DEBUG_LIMIT_TR and len(idx) > FLConfig.DEBUG_LIMIT_TR:
            idx = rng.choice(idx, size=FLConfig.DEBUG_LIMIT_TR, replace=False)

        Xtr = self.Xtr_all[idx]
        ytr = self.ytr_all[idx]

        val_limit = FLConfig.DEBUG_LIMIT_VA or len(self.yva_all)
        vsel = rng.choice(len(self.yva_all),
                         size=min(val_limit, len(self.yva_all)),
                         replace=False)
        Xva = self.Xva_all[vsel]
        yva = self.yva_all[vsel]

        self.n_train = int(len(ytr))
        self.n_val = int(len(yva))

        bs = min(FLConfig.BATCH_SIZE, max(1, self.n_train // 2))

        self.train_ds, self.val_ds = make_tf_datasets(
            Xtr, ytr, Xva, yva,
            self.input_H, self.input_W, self.input_C, bs
        )

        FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        self.hist_path = FLConfig.DRIVE_DIR / f"client_{cid}.json"
        self.eval_path = FLConfig.DRIVE_DIR / f"client_{cid}_eval.json"

        self.cid = cid

        print("\n[Client Info]", flush=True)
        print("  TF:", tf.__version__, flush=True)
        print("  GPUs:", tf.config.list_physical_devices("GPU"), flush=True)
        print("  CID:", cid, flush=True)
        print(f"  Train size: {self.n_train} Val size: {self.n_val} Batch: {bs}", flush=True)
        print("  Model input:", getattr(self.model, "input_shape", None), flush=True)

        if len(ytr) > 0:
            unique_classes, class_counts = np.unique(ytr, return_counts=True)
            class_dist = dict(zip(unique_classes, class_counts))
            print(f"  Training class distribution: {class_dist}", flush=True)

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        if parameters:

                self.model.set_weights(parameters)


        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)

        callbacks = [
            PrintMetrics(cid=str(cid), server_round=server_round),
            SaveHistory(json_path=self.hist_path, cid=str(cid), server_round=server_round),
            LastModelSaver(cid=cid),
        ]


        hist = self.model.fit(
                self.train_ds,
                epochs=FLConfig.LOCAL_EPOCHS,
                verbose=1,
                validation_data=self.val_ds,
                callbacks=callbacks,
            )


        self.last_local_weights = [w.copy() for w in self.model.get_weights()]

        metrics = {"final_accuracy": float(hist.history.get("accuracy", [0.0])[-1])}

        gc.collect()

        return self.model.get_weights(), self.n_train, metrics



    def evaluate(self, parameters, config):

        if parameters:
            self.model.set_weights(parameters)

        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)


        g_loss, g_acc = self.model.evaluate(self.val_ds, verbose=0)

        l_loss = l_acc = None
        global_snapshot = [w.copy() for w in self.model.get_weights()]

        try:
            if self.last_local_weights is not None:
                self.model.set_weights(self.last_local_weights)
                l_loss, l_acc = self.model.evaluate(self.val_ds, verbose=0)
        finally:
            self.model.set_weights(global_snapshot)


        payload = {"cid": str(cid)}
        if self.eval_path.exists():

                with open(self.eval_path, "r") as f:
                    existing = json.load(f)
                    if isinstance(existing, dict):
                        payload.update(existing)


        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()

        records = list(payload.get("eval", []))
        records.append({
            "timestamp": now_iso(),
            "server_round": server_round,
            "scope": "global",
            "val_loss": float(g_loss),
            "val_accuracy": float(g_acc),
            "n_val": int(self.n_val),
        })

        if l_loss is not None and l_acc is not None:
            records.append({
                "timestamp": now_iso(),
                "server_round": server_round,
                "scope": "local",
                "val_loss": float(l_loss),
                "val_accuracy": float(l_acc),
                "n_val": int(self.n_val),
            })

        payload["eval"] = records

        with open(self.eval_path, "w") as f:
            json.dump(payload, f, indent=2)


        return float(g_loss), self.n_val, {"val_accuracy": float(g_acc)}


class StrategyWithRound(FedAvg):
    def __init__(self, num_logical_clients: int, round_offset: int = 0, **kwargs):
        super().__init__(**kwargs)
        self.num_logical_clients = int(num_logical_clients)
        self._cid_map = {}
        self._next = 0
        self.round_offset = int(round_offset)

        self.best_accuracy, self.best_round = read_best_model_info()

    def _logical_cid(self, sim_cid: str) -> int:
        if sim_cid not in self._cid_map:
            lid = min(self._next, self.num_logical_clients - 1)
            self._cid_map[sim_cid] = lid
            self._next = min(self._next + 1, self.num_logical_clients - 1)
        return self._cid_map[sim_cid]

    def configure_fit(self, server_round, parameters, client_manager):
        cfg = super().configure_fit(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, fit_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(fit_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, FitIns(fit_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")

        print(f"[Strategy] Round {global_round} (FL round {server_round}): FIT map {human}", flush=True)
        return wrapped

    def configure_evaluate(self, server_round, parameters, client_manager):
        cfg = super().configure_evaluate(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, eval_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(eval_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, EvaluateIns(eval_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")

        if wrapped:
            print(f"[Strategy] Round {global_round} (FL round {server_round}): EVAL map {human}", flush=True)
        else:
            print(f"[Strategy] Round {global_round} (FL round {server_round}): EVAL none selected", flush=True)

        return wrapped

    def initialize_parameters(self, client_manager):
        if FLConfig.GLOBAL_LATEST_NPZ.exists():

                nds = load_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ)
                print(f"[Resume] Loaded global_latest from {FLConfig.GLOBAL_LATEST_NPZ}", flush=True)
                return ndarrays_to_parameters(nds)


        return None

    def aggregate_fit(self, server_round, results, failures):
        agg = super().aggregate_fit(server_round, results, failures)
        if agg is None:
            return agg


        global_round = server_round + self.round_offset

        parameters, metrics = agg
        if parameters is not None:

                nds = parameters_to_ndarrays(parameters)


                save_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ, nds)
                model = load_autokeras_model_safely(FLConfig.MODEL_PATH)
                model.set_weights(nds)
                model.save(FLConfig.GLOBAL_LATEST_KERAS, overwrite=True)

                print(f"[Save] Global weights saved for round {global_round}", flush=True)



        return parameters, metrics

    def aggregate_evaluate(self, server_round, results, failures):
        agg = super().aggregate_evaluate(server_round, results, failures)
        if agg is None:
            return agg


        global_round = server_round + self.round_offset

        loss, metrics = agg


        current_accuracy = 0.0
        if isinstance(metrics, dict):

            for key in ['accuracy', 'val_accuracy', 'acc']:
                if key in metrics:
                    current_accuracy = float(metrics[key])
                    break

        print(f"[Best Model Debug] Round {global_round}: extracted accuracy={current_accuracy:.4f}, full metrics={metrics}")

        if global_round > 0 and current_accuracy > self.best_accuracy:
            self.best_accuracy = current_accuracy
            self.best_round = global_round

            print(f"[Best Model] New best accuracy: {current_accuracy:.4f} at round {global_round}")



            if FLConfig.GLOBAL_LATEST_NPZ.exists():

                    import shutil
                    shutil.copy2(FLConfig.GLOBAL_LATEST_NPZ, FLConfig.GLOBAL_BEST_NPZ)

                    if FLConfig.GLOBAL_LATEST_KERAS.exists():
                        shutil.copy2(FLConfig.GLOBAL_LATEST_KERAS, FLConfig.GLOBAL_BEST_KERAS)


                    best_meta = {
                        "best_accuracy": float(current_accuracy),
                        "best_round": int(global_round),
                        "best_loss": float(loss),
                        "saved_at": now_iso()
                    }

                    with open(FLConfig.GLOBAL_BEST_META, "w") as f:
                        json.dump(best_meta, f, indent=2)

                    print(f"[Best Model] Saved best global model from round {global_round}")


        elif global_round > 0:
            print(f"[Best Model] Round {global_round} acc={current_accuracy:.4f} did not beat best acc={self.best_accuracy:.4f}")


        if self.best_accuracy > 0:
            write_server_state(global_round, self.best_accuracy, self.best_round)
        else:
            write_server_state(global_round)

        return loss, metrics


def get_central_evaluate_fn(round_offset: int = 0):
    (_, _), (Xva, yva), (Xte, yte) = load_numpy_tissuemnist(FLConfig.DATA_DIR)

    use_X, use_y, name = (Xte, yte, "TEST") if (Xte is not None and yte is not None) else (Xva, yva, "VAL")

    def build_server_ds(H, W, C):
        AUTOTUNE = tf.data.AUTOTUNE

        @tf.autograph.experimental.do_not_convert
        def preprocess(img, label):
            img = tf.image.resize(img, (H, W))
            img = tf.cast(img, tf.float32) / 255.0
            c_in = tf.shape(img)[-1]
            img = tf.cond(
                tf.logical_and(tf.equal(C, 3), tf.equal(c_in, 1)),
                lambda: tf.tile(img, [1, 1, 3]),
                lambda: img,
            )
            return img, tf.cast(label, tf.int32)

        def build(x, y):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
            ds = ds.batch(FLConfig.BATCH_SIZE).prefetch(AUTOTUNE)
            return ds

        return build

    print(f"[ServerEval] Using {name} set for centralized evaluation.", flush=True)

    def evaluate_fn(server_round: int, parameters_ndarrays, config):
        global_round = server_round + round_offset

        model = load_autokeras_model_safely(FLConfig.MODEL_PATH)

        try:
            _, H, W, C = model.input_shape
        except Exception:
            H = W = 28
            C = 1

        model.set_weights(parameters_ndarrays)

        ds_fn = build_server_ds(int(H), int(W), int(C))
        ds = ds_fn(use_X, use_y)

        loss, acc = model.evaluate(ds, verbose=0)
        print(f"[ServerEval] r{global_round} {name}: loss={loss:.4f} acc={acc:.4f}", flush=True)

        srv_path = FLConfig.DRIVE_DIR / "server_eval.json"
        srv_path.parent.mkdir(parents=True, exist_ok=True)

        payload = {}

        if srv_path.exists():

                with open(srv_path, "r") as f:
                    payload = json.load(f)


        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()

        rows = list(payload.get("eval", []))
        rows.append({
            "timestamp": now_iso(),
            "server_round": int(global_round),
            "dataset": name,
            "loss": float(loss),
            "accuracy": float(acc),
        })
        payload["eval"] = rows

        with open(srv_path, "w") as f:
            json.dump(payload, f, indent=2)

        return float(loss), {"accuracy": float(acc)}

    return evaluate_fn


def client_fn(context: Context):
    return TFClient().to_client()


class PostHocEvaluator:

    def __init__(self, config: FLConfig = None):
        self.config = config or FLConfig
        self.config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    def load_latest_round_and_weights(self) -> Tuple[int, Optional[List[np.ndarray]]]:

        if self.config.GLOBAL_LATEST_NPZ.exists():

                weights = load_ndarrays_npz(self.config.GLOBAL_LATEST_NPZ)
                round_num = read_last_round()
                return round_num, weights

        return 0, None

    def load_best_model_and_weights(self) -> Tuple[int, Optional[List[np.ndarray]], float]:
        if self.config.GLOBAL_BEST_NPZ.exists():

                weights = load_ndarrays_npz(self.config.GLOBAL_BEST_NPZ)

                best_acc, best_round = 0.0, 0
                if self.config.GLOBAL_BEST_META.exists():
                    with open(self.config.GLOBAL_BEST_META, "r") as f:
                        meta = json.load(f)
                        best_acc = float(meta.get("best_accuracy", 0.0))
                        best_round = int(meta.get("best_round", 0))

                return best_round, weights, best_acc

        return 0, None, 0.0

    def ensure_val_splits(self, y_val: np.ndarray) -> Path:
        split_dir = self.config.DATA_DIR / f"val_splits_equal_class_seed{self.config.SPLIT_SEED}"
        split_dir.mkdir(parents=True, exist_ok=True)

        need = any(not (split_dir / f"val_idx_client{cid}.npy").exists()
                  for cid in range(self.config.NUM_CLIENTS))

        if need:
            parts = make_equal_class_splits(y_val, self.config.NUM_CLIENTS, self.config.SPLIT_SEED)
            for cid, idx in enumerate(parts):
                np.save(split_dir / f"val_idx_client{cid}.npy", idx)

        return split_dir

    def run_evaluation(self, use_best_model: bool = True):
        print("\n" + "="*60)
        print("POST-HOC EVALUATION")
        print("="*60)


        (_, _), (Xva, yva), _ = load_numpy_tissuemnist(self.config.DATA_DIR)


        if use_best_model:
            server_round, weights, best_acc = self.load_best_model_and_weights()
            model_type = "BEST"
            print(f"[INFO] Evaluating BEST model from round {server_round} (acc: {best_acc:.4f})")
        else:
            server_round, weights = self.load_latest_round_and_weights()
            model_type = "LATEST"
            print(f"[INFO] Evaluating LATEST model from round {server_round}")

        if weights is None:
            print(f"[ERROR] No {model_type.lower()} model weights found!")
            return


        val_split_dir = self.ensure_val_splits(yva)


        model = load_autokeras_model_safely(str(self.config.MODEL_PATH))

        try:
            _, H, W, C = model.input_shape
        except Exception:
            H = W = 28
            C = 1


            model.set_weights(weights)
            print(f"[INFO] Loaded {model_type.lower()} model weights")


        report_name = f"{model_type.lower()}_model_evaluation_round_{server_round}.json"
        report_path = self.config.REPORTS_DIR / report_name

        evaluation_report = {
            "model_type": model_type,
            "server_round": int(server_round),
            "evaluation_timestamp": now_iso(),
            "client_evaluations": []
        }

        if use_best_model:
            evaluation_report["best_accuracy"] = float(best_acc)

        print(f"\n[INFO] Detailed evaluation report will be saved to: {report_path}")
        print("="*60)


def run_federated_training():
    print("\n" + "="*60)
    print("FEDERATED LEARNING TRAINING")
    print("="*60)


    FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)


    (Xtr_all, ytr_all), _, _ = load_numpy_tissuemnist(FLConfig.DATA_DIR)
    split_dir = FLConfig.DATA_DIR / f"splits_equal_class_seed{FLConfig.SPLIT_SEED}"
    ensure_equal_class_files(ytr_all, FLConfig.NUM_CLIENTS, FLConfig.SPLIT_SEED, split_dir)
    print(f"[Driver] Using equal class splits at: {split_dir}", flush=True)


    last_done = read_last_round()
    rounds_left = max(0, FLConfig.TOTAL_ROUNDS - last_done)


    round_offset = last_done

    if FLConfig.GLOBAL_LATEST_NPZ.exists() and round_offset > 0:
        print(f"[Resume] Found global_latest at {FLConfig.GLOBAL_LATEST_NPZ} (last_round={last_done})", flush=True)
        print(f"[Resume] Resuming from round {last_done + 1} (round offset: {round_offset})", flush=True)
    else:
        print(f"[Start] Starting from round 1", flush=True)

    print(f"[Driver] Will run {rounds_left} additional round(s) to reach TOTAL_ROUNDS={FLConfig.TOTAL_ROUNDS}", flush=True)

    if rounds_left <= 0:
        print("[Driver] No additional rounds needed - training already complete!")
        return

    def weighted_average_metrics(metrics_list):
        if not metrics_list:
            return {}

        total_examples = sum([num_examples for num_examples, _ in metrics_list])
        if total_examples == 0:
            return {}

        weighted_metrics = {}
        for num_examples, metrics in metrics_list:
            weight = num_examples / total_examples
            for key, value in metrics.items():
                if key in weighted_metrics:
                    weighted_metrics[key] += weight * value
                else:
                    weighted_metrics[key] = weight * value

        return weighted_metrics

    strategy = StrategyWithRound(
        num_logical_clients=FLConfig.NUM_CLIENTS,
        round_offset=round_offset,
        fraction_fit=1.0,
        min_fit_clients=FLConfig.NUM_CLIENTS,
        min_available_clients=FLConfig.NUM_CLIENTS,
        fraction_evaluate=1.0,
        min_evaluate_clients=FLConfig.NUM_CLIENTS,
        evaluate_fn=get_central_evaluate_fn(round_offset),
        evaluate_metrics_aggregation_fn=weighted_average_metrics,
    )


    server_config = ServerConfig(num_rounds=rounds_left, round_timeout=3600)

    print("[Driver] Starting Flower simulation...", flush=True)


    _ = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=FLConfig.NUM_CLIENTS,
        client_resources={"num_cpus": 1, "num_gpus": 1},
        ray_init_args={
            "ignore_reinit_error": True,
            "runtime_env": {
                "env_vars": {
                    "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH", ""),
                    "TF_GPU_ALLOCATOR": os.environ.get("TF_GPU_ALLOCATOR", "cuda_malloc_async"),
                    "TF_FORCE_GPU_ALLOW_GROWTH": os.environ.get("TF_FORCE_GPU_ALLOW_GROWTH", "true"),
                    "TF_ENABLE_LAYOUT_OPTIMIZER": "0",
                    "TF_ENABLE_ONEDNN_OPTS": "0",
                    "TF_CPP_MIN_LOG_LEVEL": os.environ.get("TF_CPP_MIN_LOG_LEVEL", "0"),
                }
            },
            "num_cpus": 2,
            "num_gpus": 1,
        },
        strategy=strategy,
        config=server_config,
    )


    best_acc, best_round = read_best_model_info()
    if best_acc > 0:
        print(f"Best model achieved {best_acc:.4f} accuracy at round {best_round}")


def run_federated_training_with_oom_stop():

    run_federated_training()



if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "evaluate":

        use_best = len(sys.argv) < 3 or sys.argv[2] != "latest"
        evaluator = PostHocEvaluator()
        evaluator.run_evaluation(use_best_model=use_best)
    else:

        run_federated_training_with_oom_stop()


        print("\nRunning post-training evaluation on best model...")
        evaluator = PostHocEvaluator()
        evaluator.run_evaluation(use_best_model=True)


#FL EfficientNet B7 NON IID

In [ ]:
import os
import glob
import json
import site
import gc
import sys
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional, Tuple, List, Dict, Any
import numpy as np


def is_oom_error(error_msg: str) -> bool:
    oom_indicators = [
        "OutOfMemoryError",
        "killed due to the node running low on memory",
        "memory pressure (OOM)",
        "ray.exceptions.OutOfMemoryError",
        "Task was killed due to memory",
        "running low on memory"
    ]
    error_str = str(error_msg).lower()
    return any(indicator.lower() in error_str for indicator in oom_indicators)


class FLConfig:
    NUM_CLIENTS = 2
    TOTAL_ROUNDS = 5
    LOCAL_EPOCHS = 2
    BATCH_SIZE = 8
    DEBUG_LIMIT_TR = 30
    DEBUG_LIMIT_VA = 20
    SPLIT_SEED = 42
    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/tissuemnist")
    MODEL_PATH = "/content/drive/MyDrive/Colab Notebooks/tissuemnist/tissuemnist_search/best_model.keras"
    DRIVE_DIR = Path("/content/drive/MyDrive/fl_image/fl_eff_non_iid")
    REPORTS_DIR = DRIVE_DIR / "reports"
    SERVER_STATE_PATH = DRIVE_DIR / "server_state.json"
    GLOBAL_LATEST_NPZ = DRIVE_DIR / "global_latest.npz"
    GLOBAL_LATEST_KERAS = DRIVE_DIR / "global_latest.keras"
    GLOBAL_BEST_NPZ = DRIVE_DIR / "global_best.npz"
    GLOBAL_BEST_KERAS = DRIVE_DIR / "global_best.keras"
    GLOBAL_BEST_META = DRIVE_DIR / "global_best_meta.json"
    CLIENT_LAST_DIR = DRIVE_DIR / "client_last_models"
    CKPT_FILE = DRIVE_DIR / "server_ckpt.npz"
    META_FILE = DRIVE_DIR / "server_ckpt_meta.json"


def setup_environment():
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "0")
    os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
    os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
    os.environ.setdefault("TF_ENABLE_LAYOUT_OPTIMIZER", "0")
    os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

    libdirs = []
    for d in site.getsitepackages():
        libdirs += glob.glob(os.path.join(d, "nvidia", "**", "lib"), recursive=True)

    seen, out = set(), []
    for p in libdirs:
        if os.path.isdir(p) and p not in seen:
            seen.add(p)
            out.append(p)

    if out:
        cuda_path = ":".join(out)
        os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")


setup_environment()

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable
import flwr as fl
from flwr.client import NumPyClient
from flwr.server import ServerConfig
from flwr.common import Context
from flwr.server.strategy import FedAvg
from flwr.common import FitIns, EvaluateIns, ndarrays_to_parameters, parameters_to_ndarrays


def configure_tensorflow():
    for gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(gpu, True)

    tf.config.optimizer.set_jit(False)
    tf.config.optimizer.set_experimental_options({
        "layout_optimizer": False,
        "remapping": False,
        "arithmetic_optimization": False,
        "constant_folding": False,
        "disable_meta_optimizer": True,
    })


configure_tensorflow()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def save_ndarrays_npz(path: Path, arrays: List[np.ndarray]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{f"arr_{i}": a for i, a in enumerate(arrays)})


def load_ndarrays_npz(path: Path) -> List[np.ndarray]:
    with np.load(path, allow_pickle=False) as d:
        keys = sorted([k for k in d.files if k.startswith("arr_")],
                     key=lambda k: int(k.split("_")[1]))
        return [d[k] for k in keys]


def write_server_state(last_round: int, best_accuracy: float = None, best_round: int = None) -> None:
    payload = {"last_round": int(last_round), "updated_at": now_iso()}

    if best_accuracy is not None:
        payload["best_accuracy"] = float(best_accuracy)
    if best_round is not None:
        payload["best_round"] = int(best_round)

    if FLConfig.SERVER_STATE_PATH.exists():
        with open(FLConfig.SERVER_STATE_PATH, "r") as f:
            old = json.load(f)
            if isinstance(old, dict):
                old.update(payload)
                payload = old

    with open(FLConfig.SERVER_STATE_PATH, "w") as f:
        json.dump(payload, f, indent=2)


def read_last_round() -> int:
    if FLConfig.SERVER_STATE_PATH.exists():
        with open(FLConfig.SERVER_STATE_PATH, "r") as f:
            st = json.load(f)
            return int(st.get("last_round", 0))
    return 0


def read_best_model_info() -> Tuple[float, int]:
    if FLConfig.SERVER_STATE_PATH.exists():
        with open(FLConfig.SERVER_STATE_PATH, "r") as f:
            st = json.load(f)
            return float(st.get("best_accuracy", 0.0)), int(st.get("best_round", 0))
    return 0.0, 0


@register_keras_serializable(package="Custom", name="CastToFloat32")
class CastToFloat32(keras.layers.Layer):
    def call(self, x):
        return tf.cast(x, tf.float32)


def get_autokeras_custom_objects() -> Dict:
    objs = {
        "Custom>CastToFloat32": CastToFloat32,
        "CastToFloat32": CastToFloat32,
        "autokeras.keras_layers.CastToFloat32": CastToFloat32,
    }

    import autokeras as ak
    if hasattr(ak, "CUSTOM_OBJECTS") and isinstance(ak.CUSTOM_OBJECTS, dict):
        objs.update(ak.CUSTOM_OBJECTS)

    return objs


def load_autokeras_model_safely(model_path: str) -> keras.Model:
    model = keras.models.load_model(
        model_path,
        custom_objects=get_autokeras_custom_objects(),
        compile=False
    )

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=["accuracy"],
        jit_compile=False,
    )

    return model


def maybe_expand_channels(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 3:
        arr = arr[..., None]
    return arr


def load_numpy_tissuemnist(data_dir: Path) -> Tuple:
    Xtr = np.load(data_dir / "train_images.npy", mmap_mode="r")
    ytr = np.load(data_dir / "train_labels.npy").squeeze().astype(np.int32)
    Xva = np.load(data_dir / "val_images.npy", mmap_mode="r")
    yva = np.load(data_dir / "val_labels.npy").squeeze().astype(np.int32)

    Xtr = maybe_expand_channels(Xtr)
    Xva = maybe_expand_channels(Xva)

    Xte_path = data_dir / "test_images.npy"
    yte_path = data_dir / "test_labels.npy"
    Xte = yte = None

    if Xte_path.exists() and yte_path.exists():
        Xte = maybe_expand_channels(np.load(Xte_path, mmap_mode="r"))
        yte = np.load(yte_path).squeeze().astype(np.int32)

    return (Xtr, ytr), (Xva, yva), (Xte, yte)


def make_equal_class_splits(y: np.ndarray, num_clients: int, seed: int = 42) -> List[np.ndarray]:
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    classes = np.unique(y)

    client_indices = [[] for _ in range(num_clients)]

    print(f"[Equal Split] Found {len(classes)} classes: {classes}")

    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        rng.shuffle(cls_indices)

        n_samples = len(cls_indices)
        samples_per_client = n_samples // num_clients
        remainder = n_samples % num_clients

        print(f"[Equal Split] Class {cls}: {n_samples} samples, "
              f"{samples_per_client} per client, {remainder} remainder")

        start_idx = 0
        for client_id in range(num_clients):
            end_idx = start_idx + samples_per_client

            if client_id < remainder:
                end_idx += 1

            if start_idx < len(cls_indices):
                client_indices[client_id].extend(cls_indices[start_idx:end_idx])

            start_idx = end_idx

    result = []
    for client_id in range(num_clients):
        indices = np.array(client_indices[client_id], dtype=int)
        rng.shuffle(indices)
        result.append(indices)

        client_classes, client_counts = np.unique(y[indices], return_counts=True)
        class_dist = dict(zip(client_classes, client_counts))
        print(f"[Equal Split] Client {client_id}: {len(indices)} samples, "
              f"class distribution: {class_dist}")

    return result


def ensure_equal_class_files(y_train: np.ndarray, num_clients: int, seed: int, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)

    need = any(not (outdir / f"train_idx_client{cid}.npy").exists()
              for cid in range(num_clients))

    if need:
        print(f"[Equal Split] Creating equal class splits for {num_clients} clients...")
        parts = make_equal_class_splits(y_train, num_clients, seed)
        for cid, idx in enumerate(parts):
            np.save(outdir / f"train_idx_client{cid}.npy", idx)
        print(f"[Equal Split] Saved splits to {outdir}")


def make_tf_datasets(Xtr: np.ndarray, ytr: np.ndarray,
                    Xva: np.ndarray, yva: np.ndarray,
                    H: int, W: int, Cwant: int, batch_size: int) -> Tuple:
    AUTOTUNE = tf.data.AUTOTUNE

    @tf.autograph.experimental.do_not_convert
    def preprocess(img, label):
        img = tf.image.resize(img, (H, W))
        img = tf.cast(img, tf.float32) / 255.0
        c_in = tf.shape(img)[-1]
        img = tf.cond(
            tf.logical_and(tf.equal(Cwant, 3), tf.equal(c_in, 1)),
            lambda: tf.tile(img, [1, 1, 3]),
            lambda: img,
        )
        return img, tf.cast(label, tf.int32)

    def build(x, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        if shuffle:
            ds = ds.shuffle(min(8192, y.shape[0]), reshuffle_each_iteration=True)
        ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(batch_size)
        ds = ds.cache().prefetch(AUTOTUNE)
        return ds

    return build(Xtr, ytr, True), build(Xva, yva, False)


class PrintMetrics(keras.callbacks.Callback):
    def __init__(self, cid: str, server_round: int):
        super().__init__()
        self.cid = str(cid)
        self.server_round = int(server_round)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        a = float(logs.get("accuracy", 0.0))
        l = float(logs.get("loss", 0.0))
        va = float(logs.get("val_accuracy", 0.0))
        vl = float(logs.get("val_loss", 0.0))

        print(
            f"[CID {self.cid}] r{self.server_round} epoch {epoch+1} "
            f"acc={a:.4f} loss={l:.4f} val_acc={va:.4f} val_loss={vl:.4f}",
            flush=True
        )


class SaveHistory(keras.callbacks.Callback):
    def __init__(self, json_path: Path, cid: str, server_round: int):
        super().__init__()
        self.json_path = Path(json_path)
        self.cid = str(cid)
        self.server_round = int(server_round)

        self.payload = {"cid": self.cid, "history": []}
        if self.json_path.exists():
            with open(self.json_path, "r") as f:
                data = json.load(f)
                if isinstance(data, dict):
                    self.payload.update(data)

        self.payload.setdefault("created_at", now_iso())
        self.payload["updated_at"] = now_iso()

    def on_train_begin(self, logs=None):
        self.payload["run_started_at"] = now_iso()
        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        row = {
            "timestamp": now_iso(),
            "server_round": self.server_round,
            "epoch": int(epoch + 1),
            "scope": "local",
        }

        for k, v in logs.items():
            if isinstance(v, (int, float, np.floating)):
                row[k] = float(v)

        self.payload["history"].append(row)
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_train_end(self, logs=None):
        self.payload["run_finished_at"] = now_iso()
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)


class LastModelSaver(keras.callbacks.Callback):
    def __init__(self, cid: int):
        super().__init__()
        self.cid = int(cid)
        FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)
        self.last_path = FLConfig.CLIENT_LAST_DIR / f"client_{self.cid}_last.keras"

    def on_epoch_end(self, epoch, logs=None):
        self.model.save(self.last_path, overwrite=True)


class TFClient(NumPyClient):
    def __init__(self):
        configure_tensorflow()

        (self.Xtr_all, self.ytr_all), (self.Xva_all, self.yva_all), _ = \
            load_numpy_tissuemnist(FLConfig.DATA_DIR)

        self.model = load_autokeras_model_safely(FLConfig.MODEL_PATH)

        _, H, W, C = self.model.input_shape
        self.input_H = int(H)
        self.input_W = int(W)
        self.input_C = int(C)

        self.cid = None
        self.train_ds = None
        self.val_ds = None
        self.n_train = 0
        self.n_val = 0
        self.hist_path = None
        self.eval_path = None
        self.last_local_weights = None

    def _prepare_for_cid(self, cid: int):
        if self.cid == cid and self.train_ds is not None:
            return

        split_dir = FLConfig.DATA_DIR / f"splits_equal_class_seed{FLConfig.SPLIT_SEED}"
        idx_path = split_dir / f"train_idx_client{cid}.npy"

        if idx_path.exists():
            idx = np.load(idx_path)
        else:
            idx = np.arange(len(self.ytr_all))

        rng = np.random.default_rng(FLConfig.SPLIT_SEED + int(cid))

        if FLConfig.DEBUG_LIMIT_TR and len(idx) > FLConfig.DEBUG_LIMIT_TR:
            idx = rng.choice(idx, size=FLConfig.DEBUG_LIMIT_TR, replace=False)

        Xtr = self.Xtr_all[idx]
        ytr = self.ytr_all[idx]

        val_limit = FLConfig.DEBUG_LIMIT_VA or len(self.yva_all)
        vsel = rng.choice(len(self.yva_all),
                         size=min(val_limit, len(self.yva_all)),
                         replace=False)
        Xva = self.Xva_all[vsel]
        yva = self.yva_all[vsel]

        self.n_train = int(len(ytr))
        self.n_val = int(len(yva))

        bs = min(FLConfig.BATCH_SIZE, max(1, self.n_train // 2))

        self.train_ds, self.val_ds = make_tf_datasets(
            Xtr, ytr, Xva, yva,
            self.input_H, self.input_W, self.input_C, bs
        )

        FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        self.hist_path = FLConfig.DRIVE_DIR / f"client_{cid}.json"
        self.eval_path = FLConfig.DRIVE_DIR / f"client_{cid}_eval.json"

        self.cid = cid

        print("\n[Client Info]", flush=True)
        print("  TF:", tf.__version__, flush=True)
        print("  GPUs:", tf.config.list_physical_devices("GPU"), flush=True)
        print("  CID:", cid, flush=True)
        print(f"  Train size: {self.n_train} Val size: {self.n_val} Batch: {bs}", flush=True)
        print("  Model input:", getattr(self.model, "input_shape", None), flush=True)

        if len(ytr) > 0:
            unique_classes, class_counts = np.unique(ytr, return_counts=True)
            class_dist = dict(zip(unique_classes, class_counts))
            print(f"  Training class distribution: {class_dist}", flush=True)

    def get_parameters(self, config):
        return self.model.get_weights()

    def fit(self, parameters, config):
        if parameters:
            self.model.set_weights(parameters)

        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)

        callbacks = [
            PrintMetrics(cid=str(cid), server_round=server_round),
            SaveHistory(json_path=self.hist_path, cid=str(cid), server_round=server_round),
            LastModelSaver(cid=cid),
        ]

        hist = self.model.fit(
            self.train_ds,
            epochs=FLConfig.LOCAL_EPOCHS,
            verbose=1,
            validation_data=self.val_ds,
            callbacks=callbacks,
        )

        self.last_local_weights = [w.copy() for w in self.model.get_weights()]
        metrics = {"final_accuracy": float(hist.history.get("accuracy", [0.0])[-1])}
        gc.collect()

        return self.model.get_weights(), self.n_train, metrics

    def evaluate(self, parameters, config):
        if parameters:
            self.model.set_weights(parameters)

        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)

        g_loss, g_acc = self.model.evaluate(self.val_ds, verbose=0)

        l_loss = l_acc = None
        global_snapshot = [w.copy() for w in self.model.get_weights()]

        if self.last_local_weights is not None:
            self.model.set_weights(self.last_local_weights)
            l_loss, l_acc = self.model.evaluate(self.val_ds, verbose=0)
            self.model.set_weights(global_snapshot)

        payload = {"cid": str(cid)}
        if self.eval_path.exists():
            with open(self.eval_path, "r") as f:
                existing = json.load(f)
                if isinstance(existing, dict):
                    payload.update(existing)

        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()

        records = list(payload.get("eval", []))
        records.append({
            "timestamp": now_iso(),
            "server_round": server_round,
            "scope": "global",
            "val_loss": float(g_loss),
            "val_accuracy": float(g_acc),
            "n_val": int(self.n_val),
        })

        if l_loss is not None and l_acc is not None:
            records.append({
                "timestamp": now_iso(),
                "server_round": server_round,
                "scope": "local",
                "val_loss": float(l_loss),
                "val_accuracy": float(l_acc),
                "n_val": int(self.n_val),
            })

        payload["eval"] = records

        with open(self.eval_path, "w") as f:
            json.dump(payload, f, indent=2)

        return float(g_loss), self.n_val, {"val_accuracy": float(g_acc)}


class StrategyWithRound(FedAvg):
    def __init__(self, num_logical_clients: int, round_offset: int = 0, **kwargs):
        super().__init__(**kwargs)
        self.num_logical_clients = int(num_logical_clients)
        self._cid_map = {}
        self._next = 0
        self.round_offset = int(round_offset)
        self.best_accuracy, self.best_round = read_best_model_info()

    def _logical_cid(self, sim_cid: str) -> int:
        if sim_cid not in self._cid_map:
            lid = min(self._next, self.num_logical_clients - 1)
            self._cid_map[sim_cid] = lid
            self._next = min(self._next + 1, self.num_logical_clients - 1)
        return self._cid_map[sim_cid]

    def configure_fit(self, server_round, parameters, client_manager):
        cfg = super().configure_fit(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, fit_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(fit_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, FitIns(fit_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")

        print(f"[Strategy] Round {global_round} (FL round {server_round}): FIT map {human}", flush=True)
        return wrapped

    def configure_evaluate(self, server_round, parameters, client_manager):
        cfg = super().configure_evaluate(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, eval_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(eval_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, EvaluateIns(eval_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")

        if wrapped:
            print(f"[Strategy] Round {global_round} (FL round {server_round}): EVAL map {human}", flush=True)
        else:
            print(f"[Strategy] Round {global_round} (FL round {server_round}): EVAL none selected", flush=True)

        return wrapped

    def initialize_parameters(self, client_manager):
        if FLConfig.GLOBAL_LATEST_NPZ.exists():
            nds = load_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ)
            print(f"[Resume] Loaded global_latest from {FLConfig.GLOBAL_LATEST_NPZ}", flush=True)
            return ndarrays_to_parameters(nds)
        return None

    def aggregate_fit(self, server_round, results, failures):
        agg = super().aggregate_fit(server_round, results, failures)
        if agg is None:
            return agg

        global_round = server_round + self.round_offset
        parameters, metrics = agg

        if parameters is not None:
            nds = parameters_to_ndarrays(parameters)
            save_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ, nds)

            model = load_autokeras_model_safely(FLConfig.MODEL_PATH)
            model.set_weights(nds)
            model.save(FLConfig.GLOBAL_LATEST_KERAS, overwrite=True)
            print(f"[Save] Global weights saved for round {global_round}", flush=True)

        return parameters, metrics

    def aggregate_evaluate(self, server_round, results, failures):
        agg = super().aggregate_evaluate(server_round, results, failures)
        if agg is None:
            return agg

        global_round = server_round + self.round_offset
        loss, metrics = agg

        current_accuracy = 0.0
        if isinstance(metrics, dict):
            for key in ['accuracy', 'val_accuracy', 'acc']:
                if key in metrics:
                    current_accuracy = float(metrics[key])
                    break

        print(f"[Best Model Debug] Round {global_round}: extracted accuracy={current_accuracy:.4f}, full metrics={metrics}")

        if global_round > 0 and current_accuracy > self.best_accuracy:
            self.best_accuracy = current_accuracy
            self.best_round = global_round

            print(f"[Best Model] New best accuracy: {current_accuracy:.4f} at round {global_round}")

            if FLConfig.GLOBAL_LATEST_NPZ.exists():
                import shutil
                shutil.copy2(FLConfig.GLOBAL_LATEST_NPZ, FLConfig.GLOBAL_BEST_NPZ)

                if FLConfig.GLOBAL_LATEST_KERAS.exists():
                    shutil.copy2(FLConfig.GLOBAL_LATEST_KERAS, FLConfig.GLOBAL_BEST_KERAS)

                best_meta = {
                    "best_accuracy": float(current_accuracy),
                    "best_round": int(global_round),
                    "best_loss": float(loss),
                    "saved_at": now_iso()
                }

                with open(FLConfig.GLOBAL_BEST_META, "w") as f:
                    json.dump(best_meta, f, indent=2)

                print(f"[Best Model] Saved best global model from round {global_round}")
        elif global_round > 0:
            print(f"[Best Model] Round {global_round} acc={current_accuracy:.4f} did not beat best acc={self.best_accuracy:.4f}")

        if self.best_accuracy > 0:
            write_server_state(global_round, self.best_accuracy, self.best_round)
        else:
            write_server_state(global_round)

        return loss, metrics


def get_central_evaluate_fn(round_offset: int = 0):
    (_, _), (Xva, yva), (Xte, yte) = load_numpy_tissuemnist(FLConfig.DATA_DIR)
    use_X, use_y, name = (Xte, yte, "TEST") if (Xte is not None and yte is not None) else (Xva, yva, "VAL")

    def build_server_ds(H, W, C):
        AUTOTUNE = tf.data.AUTOTUNE

        @tf.autograph.experimental.do_not_convert
        def preprocess(img, label):
            img = tf.image.resize(img, (H, W))
            img = tf.cast(img, tf.float32) / 255.0
            c_in = tf.shape(img)[-1]
            img = tf.cond(
                tf.logical_and(tf.equal(C, 3), tf.equal(c_in, 1)),
                lambda: tf.tile(img, [1, 1, 3]),
                lambda: img,
            )
            return img, tf.cast(label, tf.int32)

        def build(x, y):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
            ds = ds.batch(FLConfig.BATCH_SIZE).prefetch(AUTOTUNE)
            return ds

        return build

    print(f"[ServerEval] Using {name} set for centralized evaluation.", flush=True)

    def evaluate_fn(server_round: int, parameters_ndarrays, config):
        global_round = server_round + round_offset
        model = load_autokeras_model_safely(FLConfig.MODEL_PATH)

        _, H, W, C = model.input_shape
        model.set_weights(parameters_ndarrays)

        ds_fn = build_server_ds(int(H), int(W), int(C))
        ds = ds_fn(use_X, use_y)

        loss, acc = model.evaluate(ds, verbose=0)
        print(f"[ServerEval] r{global_round} {name}: loss={loss:.4f} acc={acc:.4f}", flush=True)

        srv_path = FLConfig.DRIVE_DIR / "server_eval.json"
        srv_path.parent.mkdir(parents=True, exist_ok=True)

        payload = {}
        if srv_path.exists():
            with open(srv_path, "r") as f:
                payload = json.load(f)

        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()

        rows = list(payload.get("eval", []))
        rows.append({
            "timestamp": now_iso(),
            "server_round": int(global_round),
            "dataset": name,
            "loss": float(loss),
            "accuracy": float(acc),
        })
        payload["eval"] = rows

        with open(srv_path, "w") as f:
            json.dump(payload, f, indent=2)

        return float(loss), {"accuracy": float(acc)}

    return evaluate_fn


def client_fn(context: Context):
    return TFClient().to_client()


class PostHocEvaluator:
    def __init__(self, config: FLConfig = None):
        self.config = config or FLConfig
        self.config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)

    def load_latest_round_and_weights(self) -> Tuple[int, Optional[List[np.ndarray]]]:
        if self.config.GLOBAL_LATEST_NPZ.exists():
            weights = load_ndarrays_npz(self.config.GLOBAL_LATEST_NPZ)
            round_num = read_last_round()
            return round_num, weights
        return 0, None

    def load_best_model_and_weights(self) -> Tuple[int, Optional[List[np.ndarray]], float]:
        if self.config.GLOBAL_BEST_NPZ.exists():
            weights = load_ndarrays_npz(self.config.GLOBAL_BEST_NPZ)

            best_acc, best_round = 0.0, 0
            if self.config.GLOBAL_BEST_META.exists():
                with open(self.config.GLOBAL_BEST_META, "r") as f:
                    meta = json.load(f)
                    best_acc = float(meta.get("best_accuracy", 0.0))
                    best_round = int(meta.get("best_round", 0))

            return best_round, weights, best_acc
        return 0, None, 0.0

    def ensure_val_splits(self, y_val: np.ndarray) -> Path:
        split_dir = self.config.DATA_DIR / f"val_splits_equal_class_seed{self.config.SPLIT_SEED}"
        split_dir.mkdir(parents=True, exist_ok=True)

        need = any(not (split_dir / f"val_idx_client{cid}.npy").exists()
                  for cid in range(self.config.NUM_CLIENTS))

        if need:
            parts = make_equal_class_splits(y_val, self.config.NUM_CLIENTS, self.config.SPLIT_SEED)
            for cid, idx in enumerate(parts):
                np.save(split_dir / f"val_idx_client{cid}.npy", idx)

        return split_dir

    def run_evaluation(self, use_best_model: bool = True):
        print("\n" + "="*60)
        print("POST-HOC EVALUATION")
        print("="*60)

        (_, _), (Xva, yva), _ = load_numpy_tissuemnist(self.config.DATA_DIR)

        if use_best_model:
            server_round, weights, best_acc = self.load_best_model_and_weights()
            model_type = "BEST"
            print(f"[INFO] Evaluating BEST model from round {server_round} (acc: {best_acc:.4f})")
        else:
            server_round, weights = self.load_latest_round_and_weights()
            model_type = "LATEST"
            print(f"[INFO] Evaluating LATEST model from round {server_round}")

        if weights is None:
            print(f"[ERROR] No {model_type.lower()} model weights found!")
            return

        val_split_dir = self.ensure_val_splits(yva)
        model = load_autokeras_model_safely(str(self.config.MODEL_PATH))

        _, H, W, C = model.input_shape
        model.set_weights(weights)
        print(f"[INFO] Loaded {model_type.lower()} model weights")

        report_name = f"{model_type.lower()}_model_evaluation_round_{server_round}.json"
        report_path = self.config.REPORTS_DIR / report_name

        evaluation_report = {
            "model_type": model_type,
            "server_round": int(server_round),
            "evaluation_timestamp": now_iso(),
            "client_evaluations": []
        }

        if use_best_model:
            evaluation_report["best_accuracy"] = float(best_acc)

        print(f"\n[INFO] Detailed evaluation report will be saved to: {report_path}")
        print("="*60)


def run_federated_training():
    print("\n" + "="*60)
    print("FEDERATED LEARNING TRAINING")
    print("="*60)

    FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)

    (Xtr_all, ytr_all), _, _ = load_numpy_tissuemnist(FLConfig.DATA_DIR)
    split_dir = FLConfig.DATA_DIR / f"splits_equal_class_seed{FLConfig.SPLIT_SEED}"
    ensure_equal_class_files(ytr_all, FLConfig.NUM_CLIENTS, FLConfig.SPLIT_SEED, split_dir)
    print(f"[Driver] Using equal class splits at: {split_dir}", flush=True)

    last_done = read_last_round()
    rounds_left = max(0, FLConfig.TOTAL_ROUNDS - last_done)
    round_offset = last_done

    if FLConfig.GLOBAL_LATEST_NPZ.exists() and round_offset > 0:
        print(f"[Resume] Found global_latest at {FLConfig.GLOBAL_LATEST_NPZ} (last_round={last_done})", flush=True)
        print(f"[Resume] Resuming from round {last_done + 1} (round offset: {round_offset})", flush=True)
    else:
        print(f"[Start] Starting from round 1", flush=True)

    print(f"[Driver] Will run {rounds_left} additional round(s) to reach TOTAL_ROUNDS={FLConfig.TOTAL_ROUNDS}", flush=True)

    if rounds_left <= 0:
        print("[Driver] No additional rounds needed - training already complete!")
        return

    def weighted_average_metrics(metrics_list):
        if not metrics_list:
            return {}

        total_examples = sum([num_examples for num_examples, _ in metrics_list])
        if total_examples == 0:
            return {}

        weighted_metrics = {}
        for num_examples, metrics in metrics_list:
            weight = num_examples / total_examples
            for key, value in metrics.items():
                if key in weighted_metrics:
                    weighted_metrics[key] += weight * value
                else:
                    weighted_metrics[key] = weight * value

        return weighted_metrics

    strategy = StrategyWithRound(
        num_logical_clients=FLConfig.NUM_CLIENTS,
        round_offset=round_offset,
        fraction_fit=1.0,
        min_fit_clients=FLConfig.NUM_CLIENTS,
        min_available_clients=FLConfig.NUM_CLIENTS,
        fraction_evaluate=1.0,
        min_evaluate_clients=FLConfig.NUM_CLIENTS,
        evaluate_fn=get_central_evaluate_fn(round_offset),
        evaluate_metrics_aggregation_fn=weighted_average_metrics,
    )

    server_config = ServerConfig(num_rounds=rounds_left, round_timeout=3600)
    print("[Driver] Starting Flower simulation...", flush=True)

    _ = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=FLConfig.NUM_CLIENTS,
        client_resources={"num_cpus": 1, "num_gpus": 1},
        ray_init_args={
            "ignore_reinit_error": True,
            "runtime_env": {
                "env_vars": {
                    "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH", ""),
                    "TF_GPU_ALLOCATOR": os.environ.get("TF_GPU_ALLOCATOR", "cuda_malloc_async"),
                    "TF_FORCE_GPU_ALLOW_GROWTH": os.environ.get("TF_FORCE_GPU_ALLOW_GROWTH", "true"),
                    "TF_ENABLE_LAYOUT_OPTIMIZER": "0",
                    "TF_ENABLE_ONEDNN_OPTS": "0",
                    "TF_CPP_MIN_LOG_LEVEL": os.environ.get("TF_CPP_MIN_LOG_LEVEL", "0"),
                }
            },
            "num_cpus": 2,
            "num_gpus": 1,
        },
        strategy=strategy,
        config=server_config,
    )

    print("\nSimulation finished.")

    best_acc, best_round = read_best_model_info()
    if best_acc > 0:
        print(f"Best model achieved {best_acc:.4f} accuracy at round {best_round}")
        print(f"Best model saved at: {FLConfig.GLOBAL_BEST_KERAS}")

    print(f"Latest model saved at: {FLConfig.GLOBAL_LATEST_KERAS}")
    print(f"All checkpoints and logs saved under: {FLConfig.DRIVE_DIR}\n")


def run_federated_training_with_oom_stop():
    run_federated_training()


if __name__ == "__main__":
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == "evaluate":
        use_best = len(sys.argv) < 3 or sys.argv[2] != "latest"
        evaluator = PostHocEvaluator()
        evaluator.run_evaluation(use_best_model=use_best)
    else:
        run_federated_training_with_oom_stop()

        print("\nRunning post-training evaluation on best model...")
        evaluator = PostHocEvaluator()
        evaluator.run_evaluation(use_best_model=True)

#Centralized ResNet-18

In [ ]:
import keras
import numpy as np
from pathlib import Path
import tensorflow as tf
import time
from typing import Tuple
import matplotlib.pyplot as plt
import random
import os


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'


tf.config.experimental.enable_op_determinism()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f" GPU detected: {gpus[0].name}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)
else:
    print(" No GPU detected! Time to pray to lord because training will be slow")



tf.keras.mixed_precision.set_global_policy('mixed_float16')


DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/tissuemnist")

def load_tissuemnist(data_dir: Path) -> Tuple:
    print("\nLoading TissueMNIST dataset...")
    start = time.time()


    Xtr = np.load(data_dir / "train_images.npy")
    ytr = np.load(data_dir / "train_labels.npy").squeeze().astype(np.int32)


    Xva = np.load(data_dir / "val_images.npy")
    yva = np.load(data_dir / "val_labels.npy").squeeze().astype(np.int32)

    if Xtr.ndim == 3:
        Xtr = Xtr[..., np.newaxis]
    if Xva.ndim == 3:
        Xva = Xva[..., np.newaxis]


    Xte = yte = None
    test_images_path = data_dir / "test_images.npy"
    test_labels_path = data_dir / "test_labels.npy"
    if test_images_path.exists() and test_labels_path.exists():
        Xte = np.load(test_images_path)
        yte = np.load(test_labels_path).squeeze().astype(np.int32)
        if Xte.ndim == 3:
            Xte = Xte[..., np.newaxis]


    print(f"  Training: {Xtr.shape} - {len(np.unique(ytr))} classes")
    print(f"  Validation: {Xva.shape}")
    if Xte is not None:
        print(f"  Test: {Xte.shape}")

    return (Xtr, ytr), (Xva, yva), (Xte, yte)


class BasicBlock(keras.layers.Layer):

    def __init__(self, filters, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride

    def build(self, input_shape):
        input_filters = input_shape[-1]

        self.conv1 = keras.layers.Conv2D(
            self.filters, 3, strides=self.stride,
            padding='same', use_bias=False,
            kernel_regularizer=keras.regularizers.l2(1e-4),
            kernel_initializer=keras.initializers.HeNormal(seed=SEED)
        )
        self.bn1 = keras.layers.BatchNormalization()

        self.conv2 = keras.layers.Conv2D(
            self.filters, 3, strides=1,
            padding='same', use_bias=False,
            kernel_regularizer=keras.regularizers.l2(1e-4),
            kernel_initializer=keras.initializers.HeNormal(seed=SEED)
        )
        self.bn2 = keras.layers.BatchNormalization()
        self.use_shortcut = self.stride != 1 or input_filters != self.filters
        if self.use_shortcut:
            self.shortcut_conv = keras.layers.Conv2D(
                self.filters, 1, strides=self.stride,
                padding='same', use_bias=False,
                kernel_initializer=keras.initializers.HeNormal(seed=SEED)
            )
            self.shortcut_bn = keras.layers.BatchNormalization()

    def call(self, x, training=False):
        out = self.conv1(x)
        out = self.bn1(out, training=training)
        out = tf.nn.relu(out)

        out = self.conv2(out)
        out = self.bn2(out, training=training)


        if self.use_shortcut:
            shortcut = self.shortcut_conv(x)
            shortcut = self.shortcut_bn(shortcut, training=training)
        else:
            shortcut = x

        out = out + shortcut
        out = tf.nn.relu(out)
        return out



def FullResNet18(in_channels=1, num_classes=8):

    inputs = keras.Input(shape=(28, 28, in_channels))
    x = keras.layers.Conv2D(
        64, 3, padding='same', use_bias=False,
        kernel_initializer=keras.initializers.HeNormal(seed=SEED)
    )(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    x = BasicBlock(64, stride=1)(x)
    x = BasicBlock(64, stride=1)(x)
    x = BasicBlock(128, stride=2)(x)
    x = BasicBlock(128, stride=1)(x)
    x = BasicBlock(256, stride=2)(x)
    x = BasicBlock(256, stride=1)(x)
    x = BasicBlock(512, stride=2)(x)
    x = BasicBlock(512, stride=1)(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.5, seed=SEED)(x)
    outputs = keras.layers.Dense(
        num_classes, activation='softmax', dtype='float32',
        kernel_initializer=keras.initializers.HeNormal(seed=SEED)
    )(x)

    return keras.Model(inputs, outputs, name='FullResNet18')



(Xtr, ytr), (Xva, yva), (Xte, yte) = load_tissuemnist(DATA_DIR)



data_augmentation = keras.Sequential([
    keras.layers.RandomRotation(factor=0.15, seed=SEED),
    keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1, seed=SEED),
    keras.layers.RandomFlip(mode="horizontal_and_vertical", seed=SEED),
    keras.layers.RandomZoom(height_factor=0.1, seed=SEED),
    keras.layers.RandomContrast(factor=0.1, seed=SEED),
], name="augmentation")




Xtr = Xtr.astype(np.float32) / 255.0
Xva = Xva.astype(np.float32) / 255.0
if Xte is not None:
    Xte = Xte.astype(np.float32) / 255.0

num_classes = len(np.unique(ytr))
print(f"Number of classes: {num_classes}")

if len(ytr.shape) == 2:
    ytr = np.argmax(ytr, axis=1)
if len(yva.shape) == 2:
    yva = np.argmax(yva, axis=1)
if yte is not None and len(yte.shape) == 2:
    yte = np.argmax(yte, axis=1)

print(f"Label shapes - Train: {ytr.shape}, Val: {yva.shape}")


rng = np.random.RandomState(SEED)
shuffle_indices = rng.permutation(len(Xtr))
Xtr = Xtr[shuffle_indices]
ytr = ytr[shuffle_indices]



base_model = FullResNet18(in_channels=1, num_classes=num_classes)
BATCH_SIZE = 32
EPOCHS = 40
INITIAL_LR = 2e-5


inputs = keras.Input(shape=(28, 28, 1))
x = data_augmentation(inputs, training=True)
outputs = base_model(x)
model = keras.Model(inputs, outputs, name=f"Augmented_{base_model.name}")
print(f"Total parameters: {model.count_params():,}")
print(f"Steps per epoch: {len(Xtr) // BATCH_SIZE}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [

    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        '/content/best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )
]


_ = model.predict(Xtr[:10], verbose=0)


start_time = time.time()

train_dataset = tf.data.Dataset.from_tensor_slices((Xtr, ytr))
train_dataset = train_dataset.shuffle(buffer_size=len(Xtr), seed=SEED)
train_dataset = train_dataset.batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices((Xva, yva))
val_dataset = val_dataset.batch(BATCH_SIZE)

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time


print(f"Total training time: {training_time:.1f} seconds ({training_time/60:.1f} minutes)")
print(f"Time per epoch: {training_time/len(history.history['loss']):.1f} seconds")


best_train_acc = max(history.history['accuracy'])
best_val_acc = max(history.history['val_accuracy'])
best_val_epoch = np.argmax(history.history['val_accuracy']) + 1

print(f"\nBest Training Accuracy: {best_train_acc*100:.2f}%")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}% (epoch {best_val_epoch})")

print("\nFinal Model Performance:")
print("-" * 63)


train_loss, train_acc = model.evaluate(Xtr[:10000], ytr[:10000], verbose=0)
print(f"Training Accuracy (subset): {train_acc*100:.2f}%")

val_loss, val_acc = model.evaluate(Xva, yva, verbose=0)
print(f"Validation Accuracy: {val_acc*100:.2f}%")

if Xte is not None:
    test_loss, test_acc = model.evaluate(Xte, yte, verbose=0)
    print(f"Test Accuracy: {test_acc*100:.2f}%")

    best_model = keras.models.load_model('/content/best_model.h5')
    print("-" * 63)
    val_loss, val_acc = best_model.evaluate(Xva, yva, verbose=0)
    print(f"Best Validation Accuracy: {val_acc*100:.2f}%")
    if Xte is not None:
        test_loss, test_acc = best_model.evaluate(Xte, yte, verbose=0)
        print(f"Best Test Accuracy: {test_acc*100:.2f}%")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].axvline(x=best_val_epoch-1, color='r', linestyle='--', alpha=0.5, label=f'Best Val (epoch {best_val_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].axvline(x=best_val_epoch-1, color='r', linestyle='--', alpha=0.5, label=f'Best Val (epoch {best_val_epoch})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


model_name = 'centralized_full_resnet18'
save_path = f"/content/drive/MyDrive/Colab Notebooks/tissuemnist/{model_name}_final.h5"
model.save(save_path)
print(f"\n Final model saved to: {save_path}")

best_save_path = f"/content/drive/MyDrive/Colab Notebooks/tissuemnist/{model_name}_best.h5"
best_model.save(best_save_path)




#FL ResNet-18

In [ ]:

import os
import glob
import json
import site
import gc
import sys
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional, Tuple, List, Dict, Any
import numpy as np


def is_oom_error(error_msg: str) -> bool:

    oom_indicators = [
        "OutOfMemoryError",
        "killed due to the node running low on memory",
        "memory pressure (OOM)",
        "ray.exceptions.OutOfMemoryError",
        "Task was killed due to memory",
        "running low on memory"
    ]

    error_str = str(error_msg).lower()
    return any(indicator.lower() in error_str for indicator in oom_indicators)


class FLConfig:

    NUM_CLIENTS = 2
    TOTAL_ROUNDS = 3
    LOCAL_EPOCHS = 2
    BATCH_SIZE = 32

    DEBUG_LIMIT_TR = 30
    DEBUG_LIMIT_VA = 20

    DISTRIBUTION_TYPE = "dirichlet" # TYPE: dirichlet or equal
    DIRICHLET_ALPHA = 0.3
    SPLIT_SEED = 42
    MIN_SAMPLES_PER_CLIENT = 50


    NUM_CLASSES = 8
    INPUT_SHAPE = 1
    LEARNING_RATE = 2e-5

    USE_AUGMENTATION = True
    AUGMENTATION_STRENGTH = 1.0


    DATA_DIR = Path("/content/drive/MyDrive/Colab Notebooks/tissuemnist")
    DRIVE_DIR = Path("/content/drive/MyDrive/fl_image/fl_resnet18")
    REPORTS_DIR = DRIVE_DIR / "reports"

    SERVER_STATE_PATH = DRIVE_DIR / "server_state.json"
    GLOBAL_LATEST_NPZ = DRIVE_DIR / "global_latest.npz"
    GLOBAL_LATEST_KERAS = DRIVE_DIR / "global_latest.keras"
    GLOBAL_BEST_NPZ = DRIVE_DIR / "global_best.npz"
    GLOBAL_BEST_KERAS = DRIVE_DIR / "global_best.keras"
    GLOBAL_BEST_META = DRIVE_DIR / "global_best_meta.json"

    CLIENT_LAST_DIR = DRIVE_DIR / "client_last_models"

    CKPT_FILE = DRIVE_DIR / "server_ckpt.npz"
    META_FILE = DRIVE_DIR / "server_ckpt_meta.json"


def setup_environment():
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "0")
    os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
    os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")
    os.environ.setdefault("TF_ENABLE_LAYOUT_OPTIMIZER", "0")
    os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")

    libdirs = []
    for d in site.getsitepackages():
        libdirs += glob.glob(os.path.join(d, "nvidia", "**", "lib"), recursive=True)

    seen, out = set(), []
    for p in libdirs:
        if os.path.isdir(p) and p not in seen:
            seen.add(p)
            out.append(p)

    if out:
        cuda_path = ":".join(out)
        os.environ["LD_LIBRARY_PATH"] = cuda_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

setup_environment()


import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import flwr as fl
from flwr.client import NumPyClient
from flwr.server import ServerConfig
from flwr.common import Context
from flwr.server.strategy import FedAvg
from flwr.common import FitIns, EvaluateIns, ndarrays_to_parameters, parameters_to_ndarrays


def configure_tensorflow():
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass

    tf.keras.mixed_precision.set_global_policy('mixed_float16')


    tf.config.optimizer.set_jit(False)
    tf.config.optimizer.set_experimental_options({
            "layout_optimizer": False,
            "remapping": False,
            "arithmetic_optimization": False,
            "constant_folding": False,
            "disable_meta_optimizer": True,
        })


configure_tensorflow()


def create_augmentation_layers(strength: float = 1.0):
    return keras.Sequential([
        keras.layers.RandomRotation(factor=0.15 * strength),
        keras.layers.RandomTranslation(
            height_factor=0.1 * strength,
            width_factor=0.1 * strength
        ),
        keras.layers.RandomFlip(mode="horizontal_and_vertical"),
        keras.layers.RandomZoom(height_factor=0.1 * strength),
        keras.layers.RandomContrast(factor=0.1 * strength),
    ], name="augmentation")


class BasicBlock(keras.layers.Layer):

    def __init__(self, filters, stride=1, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride

    def build(self, input_shape):
        input_filters = input_shape[-1]
        self.conv1 = keras.layers.Conv2D(
            self.filters, 3, strides=self.stride,
            padding='same', use_bias=False,
            kernel_regularizer=keras.regularizers.l2(1e-4)
        )
        self.bn1 = keras.layers.BatchNormalization()
        self.conv2 = keras.layers.Conv2D(
            self.filters, 3, strides=1,
            padding='same', use_bias=False,
            kernel_regularizer=keras.regularizers.l2(1e-4)
        )
        self.bn2 = keras.layers.BatchNormalization()
        self.use_shortcut = self.stride != 1 or input_filters != self.filters
        if self.use_shortcut:
            self.shortcut_conv = keras.layers.Conv2D(
                self.filters, 1, strides=self.stride,
                padding='same', use_bias=False
            )
            self.shortcut_bn = keras.layers.BatchNormalization()

    def call(self, x, training=False):
        out = self.conv1(x)
        out = self.bn1(out, training=training)
        out = tf.nn.relu(out)
        out = self.conv2(out)
        out = self.bn2(out, training=training)
        if self.use_shortcut:
            shortcut = self.shortcut_conv(x)
            shortcut = self.shortcut_bn(shortcut, training=training)
        else:
            shortcut = x
        out = out + shortcut
        out = tf.nn.relu(out)
        return out

def create_resnet18_base(in_channels=1, num_classes=8):
    inputs = keras.Input(shape=(28, 28, in_channels))

    x = keras.layers.Conv2D(64, 3, padding='same', use_bias=False)(inputs)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.ReLU()(x)
    x = BasicBlock(64, stride=1)(x)
    x = BasicBlock(64, stride=1)(x)
    x = BasicBlock(128, stride=2)(x)
    x = BasicBlock(128, stride=1)(x)
    x = BasicBlock(256, stride=2)(x)
    x = BasicBlock(256, stride=1)(x)
    x = BasicBlock(512, stride=2)(x)
    x = BasicBlock(512, stride=1)(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.5)(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax', dtype='float32')(x)
    return keras.Model(inputs, outputs, name='ResNet18_Base')

def create_augmented_resnet18(in_channels=1, num_classes=8,
                              augmentation_strength=1.0,
                              use_augmentation=True):
    inputs = keras.Input(shape=(28, 28, in_channels))

    if use_augmentation:
        augmentation = create_augmentation_layers(augmentation_strength)
        x = augmentation(inputs, training=True)
        base_model = create_resnet18_base(in_channels, num_classes)
        outputs = base_model(x)
        return keras.Model(inputs, outputs, name='Augmented_ResNet18')
    else:
        return create_resnet18_base(in_channels, num_classes)

def create_compiled_resnet18(in_channels=1, num_classes=8,
                            learning_rate=1e-3,
                            use_augmentation=True,
                            augmentation_strength=1.0):
    model = create_augmented_resnet18(
        in_channels=in_channels,
        num_classes=num_classes,
        augmentation_strength=augmentation_strength,
        use_augmentation=use_augmentation
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
        metrics=['accuracy'],
        jit_compile=False,
    )

    return model


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")

def save_ndarrays_npz(path: Path, arrays: List[np.ndarray]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, **{f"arr_{i}": a for i, a in enumerate(arrays)})

def load_ndarrays_npz(path: Path) -> List[np.ndarray]:
    with np.load(path, allow_pickle=False) as d:
        keys = sorted([k for k in d.files if k.startswith("arr_")],
                     key=lambda k: int(k.split("_")[1]))
        return [d[k] for k in keys]

def write_server_state(last_round: int, best_accuracy: float = None, best_round: int = None) -> None:
    payload = {"last_round": int(last_round), "updated_at": now_iso()}

    if best_accuracy is not None:
        payload["best_accuracy"] = float(best_accuracy)
    if best_round is not None:
        payload["best_round"] = int(best_round)

    if FLConfig.SERVER_STATE_PATH.exists():
            with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                old = json.load(f)
                if isinstance(old, dict):
                    old.update(payload)
                    payload = old


    with open(FLConfig.SERVER_STATE_PATH, "w") as f:
        json.dump(payload, f, indent=2)

def read_last_round() -> int:
    if FLConfig.SERVER_STATE_PATH.exists():
        with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                st = json.load(f)
                return int(st.get("last_round", 0))
    return 0

def read_best_model_info() -> Tuple[float, int]:
    if FLConfig.SERVER_STATE_PATH.exists():
        with open(FLConfig.SERVER_STATE_PATH, "r") as f:
                st = json.load(f)
                return float(st.get("best_accuracy", 0.0)), int(st.get("best_round", 0))
    return 0.0, 0


def maybe_expand_channels(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 3:
        arr = arr[..., None]
    return arr

def load_numpy_tissuemnist(data_dir: Path) -> Tuple:
    Xtr = np.load(data_dir / "train_images.npy", mmap_mode="r")
    ytr = np.load(data_dir / "train_labels.npy").squeeze().astype(np.int32)
    Xva = np.load(data_dir / "val_images.npy", mmap_mode="r")
    yva = np.load(data_dir / "val_labels.npy").squeeze().astype(np.int32)

    Xtr = maybe_expand_channels(Xtr)
    Xva = maybe_expand_channels(Xva)

    Xte_path = data_dir / "test_images.npy"
    yte_path = data_dir / "test_labels.npy"
    Xte = yte = None

    if Xte_path.exists() and yte_path.exists():
        Xte = maybe_expand_channels(np.load(Xte_path, mmap_mode="r"))
        yte = np.load(yte_path).squeeze().astype(np.int32)

    return (Xtr, ytr), (Xva, yva), (Xte, yte)


def make_dirichlet_splits(y: np.ndarray, num_clients: int, alpha: float = 0.5,
                         seed: int = 42, min_samples: int = 50) -> Tuple[List[np.ndarray], List[dict]]:

    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    n_samples = len(y)
    classes = np.unique(y)
    n_classes = len(classes)

    print(f"\n[Dirichlet Split] Creating heterogeneous splits:")
    print(f"  Alpha: {alpha} (lower = more heterogeneous)")
    print(f"  Clients: {num_clients}")
    print(f"  Classes: {n_classes} ({classes})")
    print(f"  Total samples: {n_samples}")

    client_indices = [[] for _ in range(num_clients)]

    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        n_cls_samples = len(cls_indices)

        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        sample_counts = np.floor(proportions * n_cls_samples).astype(int)

        remainder = n_cls_samples - sample_counts.sum()
        if remainder > 0:
            top_clients = np.argsort(proportions)[-remainder:]
            sample_counts[top_clients] += 1
        elif remainder < 0:
            top_clients = np.argsort(sample_counts)[:abs(remainder)]
            sample_counts[top_clients] -= 1

        rng.shuffle(cls_indices)

        start_idx = 0
        for client_id in range(num_clients):
            end_idx = start_idx + sample_counts[client_id]
            if sample_counts[client_id] > 0:
                client_indices[client_id].extend(cls_indices[start_idx:end_idx])
            start_idx = end_idx

        print(f"  Class {cls}: {n_cls_samples} samples -> distribution: {sample_counts}")

    result = []
    stats = []

    print(f"\n[Dirichlet Split] Client Statistics:")
    for client_id in range(num_clients):
        indices = np.array(client_indices[client_id], dtype=int)
        rng.shuffle(indices)
        result.append(indices)

        if len(indices) > 0:
            client_classes, client_counts = np.unique(y[indices], return_counts=True)
            class_dist = dict(zip(client_classes.tolist(), client_counts.tolist()))

            props = client_counts / client_counts.sum()
            entropy = -np.sum(props * np.log(props + 1e-10))

            dominant_class = int(client_classes[np.argmax(client_counts)])

            stats.append({
                'client_id': client_id,
                'n_samples': len(indices),
                'class_distribution': class_dist,
                'entropy': float(entropy),
                'dominant_class': dominant_class
            })

            if len(indices) < min_samples:
                print(f"  [WARNING] Client {client_id}: only {len(indices)} samples (< {min_samples})")

            print(f"  Client {client_id}: {len(indices)} samples, "
                  f"entropy={entropy:.3f}, dominant_class={dominant_class}, "
                  f"distribution={class_dist}")

    print(f"\n[Dirichlet Split] Heterogeneity Analysis:")
    entropies = [s['entropy'] for s in stats]
    print(f"  Mean entropy: {np.mean(entropies):.3f}")
    print(f"  Std entropy: {np.std(entropies):.3f}")
    print(f"  Min entropy: {np.min(entropies):.3f} (most specialized)")
    print(f"  Max entropy: {np.max(entropies):.3f} (most diverse)")

    mean_entropy = np.mean(entropies)
    std_entropy = np.std(entropies)
    specialized = [s for s in stats if s['entropy'] < mean_entropy - std_entropy]
    if specialized:
        print(f"  Specialized clients (entropy < {mean_entropy - std_entropy:.3f}):")
        for client in specialized:
            print(f"    Client {client['client_id']}: dominant class {client['dominant_class']}")

    return result, stats

def make_equal_class_splits(y: np.ndarray, num_clients: int, seed: int = 42) -> List[np.ndarray]:

    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    classes = np.unique(y)

    print(f"\n[Equal Split] Creating IID splits:")
    print(f"  Clients: {num_clients}")
    print(f"  Classes: {len(classes)}")

    client_indices = [[] for _ in range(num_clients)]

    for cls in classes:
        cls_indices = np.where(y == cls)[0]
        rng.shuffle(cls_indices)

        n_samples = len(cls_indices)
        samples_per_client = n_samples // num_clients
        remainder = n_samples % num_clients

        start_idx = 0
        for client_id in range(num_clients):
            end_idx = start_idx + samples_per_client
            if client_id < remainder:
                end_idx += 1

            if start_idx < len(cls_indices):
                client_indices[client_id].extend(cls_indices[start_idx:end_idx])

            start_idx = end_idx

    result = []
    for client_id in range(num_clients):
        indices = np.array(client_indices[client_id], dtype=int)
        rng.shuffle(indices)
        result.append(indices)
        client_classes, client_counts = np.unique(y[indices], return_counts=True)
        class_dist = dict(zip(client_classes, client_counts))
        print(f"  Client {client_id}: {len(indices)} samples, distribution: {class_dist}")

    return result

def ensure_split_files(y_train: np.ndarray, num_clients: int, distribution_type: str,
                      alpha: float, seed: int, data_dir: Path) -> Path:

    if distribution_type == "dirichlet":
        split_dir = data_dir / f"splits_dirichlet_alpha{alpha}_seed{seed}"
        split_dir.mkdir(parents=True, exist_ok=True)

        stats_file = split_dir / "dirichlet_statistics.json"
        need_create = not stats_file.exists() or any(
            not (split_dir / f"train_idx_client{cid}.npy").exists()
            for cid in range(num_clients)
        )

        if need_create:
            parts, stats = make_dirichlet_splits(
                y_train, num_clients, alpha, seed,
                FLConfig.MIN_SAMPLES_PER_CLIENT
            )
            for cid, idx in enumerate(parts):
                np.save(split_dir / f"train_idx_client{cid}.npy", idx)
            with open(stats_file, 'w') as f:
                json.dump({
                    'distribution_type': 'dirichlet',
                    'alpha': alpha,
                    'num_clients': num_clients,
                    'seed': seed,
                    'client_stats': stats,
                    'timestamp': now_iso()
                }, f, indent=2)
    else:
        split_dir = data_dir / f"splits_equal_class_seed{seed}"
        split_dir.mkdir(parents=True, exist_ok=True)

        need_create = any(
            not (split_dir / f"train_idx_client{cid}.npy").exists()
            for cid in range(num_clients)
        )

        if need_create:
            parts = make_equal_class_splits(y_train, num_clients, seed)

            for cid, idx in enumerate(parts):
                np.save(split_dir / f"train_idx_client{cid}.npy", idx)
    return split_dir

def make_tf_datasets(Xtr: np.ndarray, ytr: np.ndarray,
                    Xva: np.ndarray, yva: np.ndarray,
                    batch_size: int) -> Tuple:
    AUTOTUNE = tf.data.AUTOTUNE

    @tf.autograph.experimental.do_not_convert
    def preprocess(img, label):
        img = tf.cast(img, tf.float32) / 255.0
        return img, tf.cast(label, tf.int32)

    def build(x, y, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((x, y))
        if shuffle:
            ds = ds.shuffle(min(8192, y.shape[0]), reshuffle_each_iteration=True)
        ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE).batch(batch_size)
        ds = ds.cache().prefetch(AUTOTUNE)
        return ds

    return build(Xtr, ytr, True), build(Xva, yva, False)


class PrintMetrics(keras.callbacks.Callback):

    def __init__(self, cid: str, server_round: int):
        super().__init__()
        self.cid = str(cid)
        self.server_round = int(server_round)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        a = float(logs.get("accuracy", 0.0))
        l = float(logs.get("loss", 0.0))
        va = float(logs.get("val_accuracy", 0.0))
        vl = float(logs.get("val_loss", 0.0))

        print(
            f"[CID {self.cid}] r{self.server_round} epoch {epoch+1} "
            f"acc={a:.4f} loss={l:.4f} val_acc={va:.4f} val_loss={vl:.4f}",
            flush=True
        )

class SaveHistory(keras.callbacks.Callback):
    def __init__(self, json_path: Path, cid: str, server_round: int,
                 distribution_info: dict = None):
        super().__init__()
        self.json_path = Path(json_path)
        self.cid = str(cid)
        self.server_round = int(server_round)
        self.distribution_info = distribution_info or {}

        self.payload = {"cid": self.cid, "history": []}
        if self.json_path.exists():

                with open(self.json_path, "r") as f:
                    data = json.load(f)
                    if isinstance(data, dict):
                        self.payload.update(data)


        if self.distribution_info:
            self.payload["distribution_info"] = self.distribution_info

        self.payload.setdefault("created_at", now_iso())
        self.payload["updated_at"] = now_iso()

    def on_train_begin(self, logs=None):
        self.payload["run_started_at"] = now_iso()
        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        row = {
            "timestamp": now_iso(),
            "server_round": self.server_round,
            "epoch": int(epoch + 1),
            "scope": "local",
        }

        for k, v in logs.items():
            if isinstance(v, (int, float, np.floating)):
                row[k] = float(v)

        self.payload["history"].append(row)
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

    def on_train_end(self, logs=None):
        self.payload["run_finished_at"] = now_iso()
        self.payload["updated_at"] = now_iso()

        with open(self.json_path, "w") as f:
            json.dump(self.payload, f, indent=2)

class LastModelSaver(keras.callbacks.Callback):
    def __init__(self, cid: int):
        super().__init__()
        self.cid = int(cid)
        FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)
        self.last_path = FLConfig.CLIENT_LAST_DIR / f"client_{self.cid}_last.keras"

    def on_epoch_end(self, epoch, logs=None):
        self.model.save(self.last_path, overwrite=True)

class TFClient(NumPyClient):

    def __init__(self):
        configure_tensorflow()

        (self.Xtr_all, self.ytr_all), (self.Xva_all, self.yva_all), _ = \
            load_numpy_tissuemnist(FLConfig.DATA_DIR)

        self.model_train = create_compiled_resnet18(
            in_channels=FLConfig.INPUT_SHAPE,
            num_classes=FLConfig.NUM_CLASSES,
            learning_rate=FLConfig.LEARNING_RATE,
            use_augmentation=FLConfig.USE_AUGMENTATION,
            augmentation_strength=FLConfig.AUGMENTATION_STRENGTH
        )

        self.model_eval = create_compiled_resnet18(
            in_channels=FLConfig.INPUT_SHAPE,
            num_classes=FLConfig.NUM_CLASSES,
            learning_rate=FLConfig.LEARNING_RATE,
            use_augmentation=False
        )
        self.cid = None
        self.train_ds = None
        self.val_ds = None
        self.n_train = 0
        self.n_val = 0
        self.hist_path = None
        self.eval_path = None
        self.last_local_weights = None
        self.distribution_info = {}

    def _prepare_for_cid(self, cid: int):
        if self.cid == cid and self.train_ds is not None:
            return
        if FLConfig.DISTRIBUTION_TYPE == "dirichlet":
            split_dir = FLConfig.DATA_DIR / f"splits_dirichlet_alpha{FLConfig.DIRICHLET_ALPHA}_seed{FLConfig.SPLIT_SEED}"
        else:
            split_dir = FLConfig.DATA_DIR / f"splits_equal_class_seed{FLConfig.SPLIT_SEED}"

        idx_path = split_dir / f"train_idx_client{cid}.npy"

        if idx_path.exists():
            idx = np.load(idx_path)
        else:
            idx = np.arange(len(self.ytr_all))

        rng = np.random.default_rng(FLConfig.SPLIT_SEED + int(cid))

        if FLConfig.DEBUG_LIMIT_TR and len(idx) > FLConfig.DEBUG_LIMIT_TR:
            idx = rng.choice(idx, size=FLConfig.DEBUG_LIMIT_TR, replace=False)

        Xtr = self.Xtr_all[idx]
        ytr = self.ytr_all[idx]

        val_limit = FLConfig.DEBUG_LIMIT_VA or len(self.yva_all)
        vsel = rng.choice(len(self.yva_all),
                         size=min(val_limit, len(self.yva_all)),
                         replace=False)
        Xva = self.Xva_all[vsel]
        yva = self.yva_all[vsel]

        self.n_train = int(len(ytr))
        self.n_val = int(len(yva))

        bs = min(FLConfig.BATCH_SIZE, max(1, self.n_train // 2))

        self.train_ds, self.val_ds = make_tf_datasets(
            Xtr, ytr, Xva, yva, bs
        )

        FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        self.hist_path = FLConfig.DRIVE_DIR / f"client_{cid}.json"
        self.eval_path = FLConfig.DRIVE_DIR / f"client_{cid}_eval.json"

        self.cid = cid

        if len(ytr) > 0:
            unique_classes, class_counts = np.unique(ytr, return_counts=True)
            class_dist = dict(zip(unique_classes.tolist(), class_counts.tolist()))
            props = class_counts / class_counts.sum()
            entropy = -np.sum(props * np.log(props + 1e-10))

            self.distribution_info = {
                'client_id': cid,
                'distribution_type': FLConfig.DISTRIBUTION_TYPE,
                'n_samples': self.n_train,
                'class_distribution': class_dist,
                'entropy': float(entropy)
            }

            if FLConfig.DISTRIBUTION_TYPE == "dirichlet":
                self.distribution_info['alpha'] = FLConfig.DIRICHLET_ALPHA
                self.distribution_info['dominant_class'] = int(unique_classes[np.argmax(class_counts)])

        print("\n[Client Info]", flush=True)
        print("  TF:", tf.__version__, flush=True)
        print("  GPUs:", tf.config.list_physical_devices("GPU"), flush=True)
        print("  CID:", cid, flush=True)
        print(f"  Train size: {self.n_train} Val size: {self.n_val} Batch: {bs}", flush=True)
        print(f"  Distribution: {FLConfig.DISTRIBUTION_TYPE}", flush=True)

        if FLConfig.DISTRIBUTION_TYPE == "dirichlet":
            print(f"  Alpha: {FLConfig.DIRICHLET_ALPHA}", flush=True)
            print(f"  Entropy: {entropy:.3f} (lower = more specialized)", flush=True)
            if 'dominant_class' in self.distribution_info:
                print(f"  Dominant class: {self.distribution_info['dominant_class']}", flush=True)

        print(f"  Class distribution: {class_dist}", flush=True)

    def get_parameters(self, config):
        return self.model_train.get_weights()

    def fit(self, parameters, config):
        if parameters:
            self.model_train.set_weights(parameters)
            self.model_eval.set_weights(parameters)


        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)

        callbacks = [
            PrintMetrics(cid=str(cid), server_round=server_round),
            SaveHistory(json_path=self.hist_path, cid=str(cid),
                       server_round=server_round,
                       distribution_info=self.distribution_info),
            LastModelSaver(cid=cid),
        ]

        hist = self.model_train.fit(
                self.train_ds,
                epochs=FLConfig.LOCAL_EPOCHS,
                verbose=1,
                validation_data=self.val_ds,
                callbacks=callbacks,
            )
        self.last_local_weights = [w.copy() for w in self.model_train.get_weights()]
        metrics = {
                "final_accuracy": float(hist.history.get("accuracy", [0.0])[-1]),
                "client_entropy": self.distribution_info.get("entropy", 0.0)
            }
        gc.collect()
        return self.model_train.get_weights(), self.n_train, metrics



    def evaluate(self, parameters, config):
        if parameters:
            self.model_train.set_weights(parameters)
            self.model_eval.set_weights(parameters)

        cid = int(str(config.get("cid_target", "0")))
        server_round = int(config.get("server_round", 0))

        self._prepare_for_cid(cid)


        g_loss, g_acc = self.model_eval.evaluate(self.val_ds, verbose=0)

        l_loss = l_acc = None
        global_snapshot = [w.copy() for w in self.model_eval.get_weights()]

        try:
            if self.last_local_weights is not None:
                self.model_eval.set_weights(self.last_local_weights)
                l_loss, l_acc = self.model_eval.evaluate(self.val_ds, verbose=0)
        finally:
            self.model_eval.set_weights(global_snapshot)


        payload = {"cid": str(cid)}
        if self.eval_path.exists():
                with open(self.eval_path, "r") as f:
                    existing = json.load(f)
                    if isinstance(existing, dict):
                        payload.update(existing)


        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()
        payload["distribution_info"] = self.distribution_info

        records = list(payload.get("eval", []))
        records.append({
            "timestamp": now_iso(),
            "server_round": server_round,
            "scope": "global",
            "val_loss": float(g_loss),
            "val_accuracy": float(g_acc),
            "n_val": int(self.n_val),
        })

        if l_loss is not None and l_acc is not None:
            records.append({
                "timestamp": now_iso(),
                "server_round": server_round,
                "scope": "local",
                "val_loss": float(l_loss),
                "val_accuracy": float(l_acc),
                "n_val": int(self.n_val),
            })

        payload["eval"] = records

        with open(self.eval_path, "w") as f:
            json.dump(payload, f, indent=2)

        return float(g_loss), self.n_val, {"val_accuracy": float(g_acc)}


class StrategyWithRound(FedAvg):

    def __init__(self, num_logical_clients: int, round_offset: int = 0, **kwargs):
        super().__init__(**kwargs)
        self.num_logical_clients = int(num_logical_clients)
        self._cid_map = {}
        self._next = 0
        self.round_offset = int(round_offset)
        self.best_accuracy, self.best_round = read_best_model_info()

    def _logical_cid(self, sim_cid: str) -> int:
        if sim_cid not in self._cid_map:
            lid = min(self._next, self.num_logical_clients - 1)
            self._cid_map[sim_cid] = lid
            self._next = min(self._next + 1, self.num_logical_clients - 1)
        return self._cid_map[sim_cid]

    def configure_fit(self, server_round, parameters, client_manager):
        cfg = super().configure_fit(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, fit_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(fit_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, FitIns(fit_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")

        print(f"[Strategy] Round {global_round} (FL round {server_round}): FIT map {human}", flush=True)
        return wrapped

    def configure_evaluate(self, server_round, parameters, client_manager):
        cfg = super().configure_evaluate(server_round, parameters, client_manager)
        wrapped = []
        human = []

        global_round = server_round + self.round_offset

        for client, eval_ins in cfg:
            sim_cid = str(getattr(client, "cid", "sim"))
            lid = self._logical_cid(sim_cid)
            new_conf = dict(eval_ins.config or {})
            new_conf["server_round"] = int(global_round)
            new_conf["cid_target"] = str(lid)
            wrapped.append((client, EvaluateIns(eval_ins.parameters, new_conf)))
            human.append(f"{sim_cid}->{lid}")
        print(f"[Strategy] Round {global_round} (FL round {server_round}): EVAL map {human}", flush=True)


    def initialize_parameters(self, client_manager):
        if FLConfig.GLOBAL_LATEST_NPZ.exists():

            nds = load_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ)
            print(f"[Resume] Loaded global_latest from {FLConfig.GLOBAL_LATEST_NPZ}", flush=True)
            return ndarrays_to_parameters(nds)
        return None

    def aggregate_fit(self, server_round, results, failures):
        agg = super().aggregate_fit(server_round, results, failures)
        if agg is None:
            return agg
        global_round = server_round + self.round_offset

        parameters, metrics = agg
        if parameters is not None:

                nds = parameters_to_ndarrays(parameters)
                save_ndarrays_npz(FLConfig.GLOBAL_LATEST_NPZ, nds)

                model = create_compiled_resnet18(
                        in_channels=FLConfig.INPUT_SHAPE,
                        num_classes=FLConfig.NUM_CLASSES,
                        learning_rate=FLConfig.LEARNING_RATE,
                        use_augmentation=False
                    )
                model.set_weights(nds)
                model.save(FLConfig.GLOBAL_LATEST_KERAS, overwrite=True)


                print(f"[Save] Global weights saved for round {global_round}", flush=True)


        return parameters, metrics

    def aggregate_evaluate(self, server_round, results, failures):
        agg = super().aggregate_evaluate(server_round, results, failures)
        if agg is None:
            return agg

        global_round = server_round + self.round_offset

        loss, metrics = agg

        current_accuracy = 0.0
        if isinstance(metrics, dict):
            for key in ['accuracy', 'val_accuracy', 'acc']:
                if key in metrics:
                    current_accuracy = float(metrics[key])
                    break

        print(f"[Best Model] Round {global_round}: accuracy={current_accuracy:.4f}")

        if global_round > 0 and current_accuracy > self.best_accuracy:
            self.best_accuracy = current_accuracy
            self.best_round = global_round

            print(f"[Best Model] New best: {current_accuracy:.4f} at round {global_round}")

            if FLConfig.GLOBAL_LATEST_NPZ.exists():
                    import shutil
                    shutil.copy2(FLConfig.GLOBAL_LATEST_NPZ, FLConfig.GLOBAL_BEST_NPZ)

                    if FLConfig.GLOBAL_LATEST_KERAS.exists():
                        shutil.copy2(FLConfig.GLOBAL_LATEST_KERAS, FLConfig.GLOBAL_BEST_KERAS)

                    best_meta = {
                        "best_accuracy": float(current_accuracy),
                        "best_round": int(global_round),
                        "best_loss": float(loss),
                        "saved_at": now_iso()
                    }

                    with open(FLConfig.GLOBAL_BEST_META, "w") as f:
                        json.dump(best_meta, f, indent=2)

                    print(f"[Best Model] Saved")



        if self.best_accuracy > 0:
            write_server_state(global_round, self.best_accuracy, self.best_round)
        else:
            write_server_state(global_round)

        return loss, metrics


def get_central_evaluate_fn(round_offset: int = 0):
    (_, _), (Xva, yva), (Xte, yte) = load_numpy_tissuemnist(FLConfig.DATA_DIR)

    use_X, use_y, name = (Xte, yte, "TEST") if (Xte is not None and yte is not None) else (Xva, yva, "VAL")

    def build_server_ds(H, W, C):
        AUTOTUNE = tf.data.AUTOTUNE

        @tf.autograph.experimental.do_not_convert
        def preprocess(img, label):
            img = tf.cast(img, tf.float32) / 255.0
            return img, tf.cast(label, tf.int32)

        def build(x, y):
            ds = tf.data.Dataset.from_tensor_slices((x, y))
            ds = ds.map(preprocess, num_parallel_calls=AUTOTUNE)
            ds = ds.batch(FLConfig.BATCH_SIZE).prefetch(AUTOTUNE)
            return ds

        return build

    def evaluate_fn(server_round: int, parameters_ndarrays, config):
        global_round = server_round + round_offset

        model = create_compiled_resnet18(
            in_channels=FLConfig.INPUT_SHAPE,
            num_classes=FLConfig.NUM_CLASSES,
            learning_rate=FLConfig.LEARNING_RATE,
            use_augmentation=False
        )

        model.set_weights(parameters_ndarrays)

        ds_fn = build_server_ds(28, 28, 1)
        ds = ds_fn(use_X, use_y)

        loss, acc = model.evaluate(ds, verbose=0)
        print(f"[ServerEval] r{global_round} {name}: loss={loss:.4f} acc={acc:.4f}", flush=True)

        srv_path = FLConfig.DRIVE_DIR / "server_eval.json"
        srv_path.parent.mkdir(parents=True, exist_ok=True)

        payload = {}

        if srv_path.exists():
                with open(srv_path, "r") as f:
                    payload = json.load(f)


        payload.setdefault("created_at", now_iso())
        payload["updated_at"] = now_iso()

        rows = list(payload.get("eval", []))
        rows.append({
            "timestamp": now_iso(),
            "server_round": int(global_round),
            "dataset": name,
            "loss": float(loss),
            "accuracy": float(acc),
        })
        payload["eval"] = rows

        with open(srv_path, "w") as f:
            json.dump(payload, f, indent=2)

        return float(loss), {"accuracy": float(acc)}

    return evaluate_fn


def client_fn(context: Context):
    return TFClient().to_client()


def run_federated_training():
    print("\n" + "="*60)
    print("FEDERATED LEARNING WITH AUGMENTED RESNET-18")
    print(f"Distribution: {FLConfig.DISTRIBUTION_TYPE.upper()}")
    if FLConfig.DISTRIBUTION_TYPE == "dirichlet":
        print(f"Dirichlet Alpha: {FLConfig.DIRICHLET_ALPHA}")
    print("="*60)
    print(f"Augmentation: {'ENABLED' if FLConfig.USE_AUGMENTATION else 'DISABLED'}")
    if FLConfig.USE_AUGMENTATION:
        print(f"Augmentation strength: {FLConfig.AUGMENTATION_STRENGTH}")
    print("="*60)

    FLConfig.DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    FLConfig.CLIENT_LAST_DIR.mkdir(parents=True, exist_ok=True)

    (Xtr_all, ytr_all), _, _ = load_numpy_tissuemnist(FLConfig.DATA_DIR)

    split_dir = ensure_split_files(
        ytr_all,
        FLConfig.NUM_CLIENTS,
        FLConfig.DISTRIBUTION_TYPE,
        FLConfig.DIRICHLET_ALPHA,
        FLConfig.SPLIT_SEED,
        FLConfig.DATA_DIR
    )

    print(f"[Driver] Using {FLConfig.DISTRIBUTION_TYPE} splits from: {split_dir}", flush=True)

    last_done = read_last_round()
    rounds_left = max(0, FLConfig.TOTAL_ROUNDS - last_done)
    round_offset = last_done

    if FLConfig.GLOBAL_LATEST_NPZ.exists() and round_offset > 0:
        print(f"[Resume] Found checkpoint at round {last_done}", flush=True)
        print(f"[Resume] Resuming from round {last_done + 1}", flush=True)
    else:
        print(f"[Start] Starting from round 1", flush=True)

    print(f"[Driver] Will run {rounds_left} round(s) to reach {FLConfig.TOTAL_ROUNDS}", flush=True)

    if rounds_left <= 0:
        print("[Driver] Training already complete!")
        return

    def weighted_average_metrics(metrics_list):
        if not metrics_list:
            return {}

        total_examples = sum([num_examples for num_examples, _ in metrics_list])
        if total_examples == 0:
            return {}

        weighted_metrics = {}
        for num_examples, metrics in metrics_list:
            weight = num_examples / total_examples
            for key, value in metrics.items():
                if key in weighted_metrics:
                    weighted_metrics[key] += weight * value
                else:
                    weighted_metrics[key] = weight * value

        return weighted_metrics

    strategy = StrategyWithRound(
        num_logical_clients=FLConfig.NUM_CLIENTS,
        round_offset=round_offset,
        fraction_fit=1.0,
        min_fit_clients=FLConfig.NUM_CLIENTS,
        min_available_clients=FLConfig.NUM_CLIENTS,
        fraction_evaluate=1.0,
        min_evaluate_clients=FLConfig.NUM_CLIENTS,
        evaluate_fn=get_central_evaluate_fn(round_offset),
        evaluate_metrics_aggregation_fn=weighted_average_metrics,
    )

    server_config = ServerConfig(num_rounds=rounds_left, round_timeout=3600)



    _ = fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=FLConfig.NUM_CLIENTS,
        client_resources={"num_cpus": 1, "num_gpus": 1},
        ray_init_args={
            "ignore_reinit_error": True,
            "runtime_env": {
                "env_vars": {
                    "LD_LIBRARY_PATH": os.environ.get("LD_LIBRARY_PATH", ""),
                    "TF_GPU_ALLOCATOR": os.environ.get("TF_GPU_ALLOCATOR", "cuda_malloc_async"),
                    "TF_FORCE_GPU_ALLOW_GROWTH": os.environ.get("TF_FORCE_GPU_ALLOW_GROWTH", "true"),
                    "TF_ENABLE_LAYOUT_OPTIMIZER": "0",
                    "TF_ENABLE_ONEDNN_OPTS": "0",
                    "TF_CPP_MIN_LOG_LEVEL": os.environ.get("TF_CPP_MIN_LOG_LEVEL", "0"),
                }
            },
            "num_cpus": 2,
            "num_gpus": 1,
        },
        strategy=strategy,
        config=server_config,
    )



    best_acc, best_round = read_best_model_info()
    if best_acc > 0:
        print(f"Best model: {best_acc:.4f} accuracy at round {best_round}")




def run_federated_training_with_oom_stop():

        run_federated_training()



if __name__ == "__main__":
    run_federated_training_with_oom_stop()